In [1]:
#!/usr/bin/env python
# Created by "Thieu" at 08:58, 16/03/2020 ----------%
#       Email: nguyenthieu2102@gmail.com            %
#       Github: https://github.com/thieu1995        %
# --------------------------------------------------%

import numpy as np
from typing import List, Union, Tuple, Dict
from mealpy.utils.agent import Agent
from mealpy.utils.problem import Problem
from math import gamma
from mealpy.utils.history import History
from mealpy.utils.target import Target
from mealpy.utils.termination import Termination
from mealpy.utils.logger import Logger
from mealpy.utils.validator import Validator
import concurrent.futures as parallel
from functools import partial
import os
import time


class Optimizer:
    """
    The base class of all algorithms. All methods in this class will be inherited

    Notes
    ~~~~~
    + The function solve() is the most important method, trained the model
    + The parallel (multithreading or multiprocessing) is used in method: generate_population(), update_target_for_population()
    + The general format of:
        + population = [agent_1, agent_2, ..., agent_N]
        + agent = [solution, target]
        + target = [fitness value, objective_list]
        + objective_list = [obj_1, obj_2, ..., obj_M]
    """

    EPSILON = 10E-10
    SUPPORTED_MODES = ["process", "thread", "swarm", "single"]
    AVAILABLE_MODES = ["process", "thread", "swarm"]
    PARALLEL_MODES = ["process", "thread"]
    SUPPORTED_ARRAYS = [list, tuple, np.ndarray]

    def __init__(self, **kwargs):
        super(Optimizer, self).__init__()
        self.epoch, self.pop_size = None, None
        self.mode, self.n_workers, self.name = None, None, None
        self.pop, self.g_best, self.g_worst = None, Agent(), None
        self.problem, self.logger, self.history = None, None, None
        self.__set_keyword_arguments(kwargs)
        self.validator = Validator(log_to="console", log_file=None)
        self.qubits = None
        self.FEs = None

        if self.name is None: self.name = self.__class__.__name__
        self.sort_flag = False
        self.nfe_counter = -1  # The first one is tested in Problem class
        self.parameters, self.params_name_ordered = {}, None
        self.is_parallelizable = True

    def __set_keyword_arguments(self, kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)

    def set_parameters(self, parameters: Union[List, Tuple, Dict]) -> None:
        """
        Set the parameters for current optimizer.

        if paras is a list of parameter's name, then it will set the default value in optimizer as current parameters
        if paras is a dict of parameter's name and value, then it will override the current parameters

        Args:
            parameters: The parameters
        """
        if type(parameters) in (list, tuple):
            self.params_name_ordered = tuple(parameters)
            self.parameters = {}
            for name in parameters:
                self.parameters[name] = self.__dict__[name]

        if type(parameters) is dict:
            valid_para_names = set(self.parameters.keys())
            new_para_names = set(parameters.keys())
            if new_para_names.issubset(valid_para_names):
                for key, value in parameters.items():
                    setattr(self, key, value)
                    self.parameters[key] = value
            else:
                raise ValueError(f"Invalid input parameters: {new_para_names} for {self.get_name()} optimizer. "
                                 f"Valid parameters are: {valid_para_names}.")

    def get_parameters(self) -> Dict:
        """
        Get parameters of optimizer.
        """
        return self.parameters

    def get_attributes(self) -> Dict:
        """
        Get all attributes in optimizer.
        """
        return self.__dict__

    def get_name(self) -> str:
        """
        Get name of the optimizer
        """
        return self.name

    def __str__(self):
        temp = ""
        for key in self.params_name_ordered:
            temp += f"{key}={self.parameters[key]}, "
        temp = temp[:-2]
        return f"{self.__class__.__name__}({temp})"

    def initialize_variables(self):
        pass

    def before_initialization(self, starting_solutions: Union[List, Tuple, np.ndarray] = None) -> None:
        """
        Args:
            starting_solutions: The starting solutions (not recommended)
        """
        if starting_solutions is None:
            pass
        elif type(starting_solutions) in self.SUPPORTED_ARRAYS and len(starting_solutions) == self.pop_size:
            if type(starting_solutions[0]) in self.SUPPORTED_ARRAYS and len(starting_solutions[0]) == self.problem.n_dims:
                self.pop = [self.generate_agent(solution) for solution in starting_solutions]
            else:
                raise ValueError("Invalid starting_solutions. It should be a list of positions or 2D matrix of positions only.")
        else:
            raise ValueError("Invalid starting_solutions. It should be a list/2D matrix of positions with same length as pop_size.")

    def initialization(self) -> None:
        if self.pop is None:
            self.pop = self.generate_population(self.pop_size)

    def after_initialization(self) -> None:
        # The initial population is sorted or not depended on algorithm's strategy
        pop_temp, best, worst = self.get_special_agents(self.pop, n_best=1, n_worst=1, minmax=self.problem.minmax)
        self.g_best, self.g_worst = best[0], worst[0]
        if self.sort_flag: self.pop = pop_temp
        ## Store initial best and worst solutions
        self.history.store_initial_best_worst(self.g_best, self.g_worst)

    def before_main_loop(self):
        pass

    def evolve(self, epoch: int) -> None:
        pass

    def check_problem(self, problem, seed) -> None:
        if isinstance(problem, Problem):
            problem.set_seed(seed)
            self.problem = problem
        elif type(problem) == dict:
            problem["seed"] = seed
            self.problem = Problem(**problem)
        else:
            raise ValueError("problem needs to be a dict or an instance of Problem class.")
        self.generator = np.random.default_rng(seed)
        self.logger = Logger(self.problem.log_to, log_file=self.problem.log_file).create_logger(name=f"{self.__module__}.{self.__class__.__name__}")
        self.logger.info(self.problem.msg)
        self.history = History(log_to=self.problem.log_to, log_file=self.problem.log_file)
        self.pop, self.g_best, self.g_worst = None, None, None

    def get_FEs(self, FEs):
        self.FEs = FEs
    

    def solve(self, problem: Union[Dict, Problem] = None, mode: str = 'single', n_workers: int = None,
              termination: Union[Dict, Termination] = None, starting_solutions: Union[List, np.ndarray, Tuple] = None,
              seed: int = None, qubits: int = None) -> Agent:
        """
        Args:
            problem: an instance of Problem class or a dictionary
            mode: Parallel: 'process', 'thread'; Sequential: 'swarm', 'single'.

                * 'process': The parallel mode with multiple cores run the tasks
                * 'thread': The parallel mode with multiple threads run the tasks
                * 'swarm': The sequential mode that no effect on updating phase of other agents
                * 'single': The sequential mode that effect on updating phase of other agents, this is default mode

            n_workers: The number of workers (cores or threads) to do the tasks (effect only on parallel mode)
            termination: The termination dictionary or an instance of Termination class
            starting_solutions: List or 2D matrix (numpy array) of starting positions with length equal pop_size parameter
            seed: seed for random number generation needed to be *explicitly* set to int value

        Returns:
            g_best: g_best, the best found agent, that hold the best solution and the best target. Access by: .g_best.solution, .g_best.target
        """
        self.check_problem(problem, seed)
        self.initialize_variables()

        self.before_initialization(starting_solutions)
        self.initialization()
        self.after_initialization()

        self.before_main_loop()
        epoch = 1
        #for epoch in range(1, self.epoch + 1):
        pop_temp, self.g_best = self.update_global_best_agent(self.pop)

        while epoch < self.epoch and qubits -1 + self.g_best.target.fitness > 1e-1:
            time_epoch = time.perf_counter()

            ## Evolve method will be called in child class
            self.evolve(epoch)

            # Update global best solution, the population is sorted or not depended on algorithm's strategy
            pop_temp, self.g_best = self.update_global_best_agent(self.pop)
            if self.sort_flag: self.pop = pop_temp

            time_epoch = time.perf_counter() - time_epoch
            if epoch % 50 == 1:
                self.track_optimize_step(self.pop, epoch, time_epoch)
                print(self.FEs)
            epoch += 1
            
        self.track_optimize_process()
        print("Number of function evaluations: ",self.FEs, "for qubits: ", qubits)
        return self.g_best

    def track_optimize_step(self, pop: List[Agent] = None, epoch: int = None, runtime: float = None) -> None:
        """
        Save some historical data and print out the detailed information of training process in each epoch

        Args:
            pop: the current population
            epoch: current iteration
            runtime: the runtime for current iteration
        """
        ## Save history data
        if self.problem.save_population:
            self.history.list_population.append(Optimizer.duplicate_pop(pop))
        self.history.list_epoch_time.append(runtime)
        self.history.list_global_best_fit.append(self.history.list_global_best[-1].target.fitness)
        self.history.list_current_best_fit.append(self.history.list_current_best[-1].target.fitness)
        # Save the exploration and exploitation data for later usage
        pos_matrix = np.array([agent.solution for agent in pop])
        div = np.mean(np.abs(np.median(pos_matrix, axis=0) - pos_matrix), axis=0)
        self.history.list_diversity.append(np.mean(div, axis=0))
        ## Print epoch
        self.logger.info(f">>>Problem: {self.problem.name}, Epoch: {epoch}, Current best: {self.history.list_current_best[-1].target.fitness}, "
                         f"Global best: {self.history.list_global_best[-1].target.fitness}, Runtime: {runtime:.5f} seconds")

    def track_optimize_process(self) -> None:
        """
        Save some historical data after training process finished
        """
        self.history.epoch = len(self.history.list_diversity)
        div_max = np.max(self.history.list_diversity)
        self.history.list_exploration = 100 * (np.array(self.history.list_diversity) / div_max)
        self.history.list_exploitation = 100 - self.history.list_exploration
        self.history.list_global_best = self.history.list_global_best[1:]
        self.history.list_current_best = self.history.list_current_best[1:]
        self.history.list_global_worst = self.history.list_global_worst[1:]
        self.history.list_current_worst = self.history.list_current_worst[1:]

    def generate_empty_agent(self, solution: np.ndarray = None) -> Agent:
        """
        Generate new agent with solution

        Args:
            solution (np.ndarray): The solution
        """
        if solution is None:
            solution = self.problem.generate_solution(encoded=True)
        return Agent(solution=solution)

    def generate_agent(self, solution: np.ndarray = None) -> Agent:
        """
        Generate new agent with full information

        Args:
            solution (np.ndarray): The solution
        """
        agent = self.generate_empty_agent(solution)
        agent.target = self.get_target(agent.solution)
        return agent

    def generate_population(self, pop_size: int = None) -> List[Agent]:
        """
        Args:
            pop_size (int): number of solutions

        Returns:
            list: population or list of solutions/agents
        """
        if pop_size is None:
            pop_size = self.pop_size
        pop = []
        if self.mode == "thread":
            with parallel.ThreadPoolExecutor(self.n_workers) as executor:
                list_executors = [executor.submit(self.generate_agent) for _ in range(pop_size)]
                # This method yield the result everytime a thread finished their job (not by order)
                for f in parallel.as_completed(list_executors):
                    pop.append(f.result())
        elif self.mode == "process":
            with parallel.ProcessPoolExecutor(self.n_workers) as executor:
                list_executors = [executor.submit(self.generate_agent) for _ in range(pop_size)]
                # This method yield the result everytime a cpu finished their job (not by order).
                for f in parallel.as_completed(list_executors):
                    pop.append(f.result())
        else:
            pop = [self.generate_agent() for _ in range(0, pop_size)]
        return pop

    def amend_solution(self, solution: np.ndarray) -> np.ndarray:
        """
        This function is based on optimizer's strategy.
        In each optimizer, this function can be overridden

        Args:
            solution: The position

        Returns:
            The valid solution based on optimizer's strategy
        """
        return solution

    def correct_solution(self, solution: np.ndarray) -> np.ndarray:
        """
        This function is based on optimizer's strategy and problem-specific condition
        DO NOT override this function

        Args:
            solution: The position

        Returns:
            The correct solution that can be used to calculate target
        """
        solution = self.amend_solution(solution)
        return self.problem.correct_solution(solution)

    def update_target_for_population(self, pop: List[Agent] = None) -> List[Agent]:
        """
        Update target for the input population

        Args:
            pop: the population of agents

        Returns:
            list: population with updated target value
        """
        pos_list = [agent.solution for agent in pop]
        if self.mode == "thread":
            with parallel.ThreadPoolExecutor(self.n_workers) as executor:
                # Return result as original order, not the future object
                list_results = executor.map(partial(self.get_target, counted=False), pos_list)
                for idx, target in enumerate(list_results):
                    pop[idx].target = target
        elif self.mode == "process":
            with parallel.ProcessPoolExecutor(self.n_workers) as executor:
                # Return result as original order, not the future object
                list_results = executor.map(partial(self.get_target, counted=False), pos_list)
                for idx, target in enumerate(list_results):
                    pop[idx].target = target
        elif self.mode == "swarm":
            for idx, pos in enumerate(pos_list):
                pop[idx].target = self.get_target(pos, counted=False)
        else:
            return pop
        self.nfe_counter += len(pop)
        return pop

    def get_target(self, solution: np.ndarray, counted: bool = True) -> Target:
        """
        Get target value

        Args:
            solution: The real-value solution
            counted: Indicating the number of function evaluations is increasing or not

        Returns:
            The target value
        """
        if counted:
            self.nfe_counter += 1
        return self.problem.get_target(solution)

    @staticmethod
    def compare_target(target_x: Target, target_y: Target, minmax: str = "min") -> bool:
        if minmax == "min":
            return True if target_x.fitness < target_y.fitness else False
        else:
            return False if target_x.fitness < target_y.fitness else True

    @staticmethod
    def compare_fitness(fitness_x: Union[float, int], fitness_y: Union[float, int], minmax: str = "min") -> bool:
        if minmax == "min":
            return True if fitness_x < fitness_y else False
        else:
            return False if fitness_x < fitness_y else True

    @staticmethod
    def duplicate_pop(pop: List[Agent]) -> List[Agent]:
        return [agent.copy() for agent in pop]

    @staticmethod
    def get_sorted_population(pop: List[Agent], minmax: str = "min", return_index: bool = False) -> List[Agent]:
        """
        Get sorted population based on type (minmax) of problem

        Args:
            pop: The population
            minmax: The type of the problem
            return_index: Return the sorted index or not

        Returns:
            Sorted population (1st agent is the best, last agent is the worst
            Sorted index (Optional)
        """

        list_fits = [agent.target.fitness for agent in pop]
        indices = np.argsort(list_fits).tolist()
        if minmax == "max":
            indices = indices[::-1]
        pop_new = [pop[idx] for idx in indices]
        if return_index:
            return pop_new, indices
        else:
            return pop_new

    @staticmethod
    def get_best_agent(pop: List[Agent], minmax: str = "min") -> Agent:
        """
        Args:
            pop: The population of agents
            minmax: The type of problem

        Returns:
            The best agent
        """
        pop = Optimizer.get_sorted_population(pop, minmax)
        return pop[0].copy()

    @staticmethod
    def get_index_best(pop: List[Agent], minmax: str = "min") -> int:
        fit_list = np.array([agent.target.fitness for agent in pop])
        if minmax == "min":
            return np.argmin(fit_list)
        else:
            return np.argmax(fit_list)

    @staticmethod
    def get_worst_agent(pop: List[Agent], minmax: str = "min") -> Agent:
        """
        Args:
            pop: The population of agents
            minmax: The type of problem

        Returns:
            The worst agent
        """
        pop = Optimizer.get_sorted_population(pop, minmax)
        return pop[-1].copy()

    @staticmethod
    def get_special_agents(pop: List[Agent] = None, n_best: int = 3, n_worst: int = 3,
                           minmax: str = "min") -> Tuple[List[Agent], Union[List[Agent], None], Union[List[Agent], None]]:
        """
        Get special agents include sorted population, n1 best agents, n2 worst agents

        Args:
            pop: The population
            n_best: Top n1 best agents, default n1=3, good level reduction
            n_worst: Top n2 worst agents, default n2=3, worst level reduction
            minmax: The problem type

        Returns:
            The sorted_population, n1 best agents and n2 worst agents
        """
        pop = Optimizer.get_sorted_population(pop, minmax)
        if n_best is None:
            if n_worst is None:
                return pop, None, None
            else:
                return pop, None, [agent.copy() for agent in pop[::-1][:n_worst]]
        else:
            if n_worst is None:
                return pop, [agent.copy() for agent in pop[:n_best]], None
            else:
                return pop, [agent.copy() for agent in pop[:n_best]], [agent.copy() for agent in pop[::-1][:n_worst]]

    @staticmethod
    def get_special_fitness(pop: List[Agent] = None, minmax: str = "min") -> Tuple[Union[float, np.ndarray], float, float]:
        """
        Get special target include the total fitness, the best fitness, and the worst fitness

        Args:
            pop: The population
            minmax: The problem type

        Returns:
            The total fitness, the best fitness, and the worst fitness
        """
        total_fitness = np.sum([agent.target.fitness for agent in pop])
        pop = Optimizer.get_sorted_population(pop, minmax)
        return total_fitness, pop[0].target.fitness, pop[-1].target.fitness

    @staticmethod
    def get_better_agent(agent_x: Agent, agent_y: Agent, minmax: str = "min", reverse: bool = False) -> Agent:
        """
        Args:
            agent_x: First agent
            agent_y: Second agent
            minmax: The problem type
            reverse: Reverse the minmax

        Returns:
            The better agent based on fitness
        """
        minmax_dict = {"min": 0, "max": 1}
        idx = minmax_dict[minmax]
        if reverse:
            idx = 1 - idx
        if idx == 0:
            return agent_x.copy() if agent_x.target.fitness < agent_y.target.fitness else agent_y.copy()
        else:
            return agent_y.copy() if agent_x.target.fitness < agent_y.target.fitness else agent_x.copy()

    ### Survivor Selection
    @staticmethod
    def greedy_selection_population(pop_old: List[Agent] = None, pop_new: List[Agent] = None, minmax: str = "min") -> List[Agent]:
        """
        Args:
            pop_old: The current population
            pop_new: The next population
            minmax: The problem type

        Returns:
            The new population with better solutions
        """
        len_old, len_new = len(pop_old), len(pop_new)
        if len_old != len_new:
            raise ValueError("Greedy selection of two population with different length.")
        if minmax == "min":
            return [pop_new[idx] if pop_new[idx].target.fitness < pop_old[idx].target.fitness else pop_old[idx] for idx in range(len_old)]
        else:
            return [pop_new[idx] if pop_new[idx].target.fitness > pop_old[idx].target.fitness else pop_old[idx] for idx in range(len_old)]

    @staticmethod
    def get_sorted_and_trimmed_population(pop: List[Agent] = None, pop_size: int = None, minmax: str = "min") -> List[Agent]:
        """
        Args:
            pop: The population
            pop_size: The number of selected agents
            minmax: The problem type

        Returns:
            The sorted and trimmed population with pop_size size
        """
        pop = Optimizer.get_sorted_population(pop, minmax)
        return pop[:pop_size]

    def update_global_best_agent(self, pop: List[Agent], save: bool = True) -> Union[List, Tuple]:
        """
        Update global best and current best solutions in history object.
        Also update global worst and current worst solutions in history object.

        Args:
            pop (list): The population of pop_size individuals
            save (bool): True if you want to add new current/global best to history, False if you just want to update current/global best

        Returns:
            list: Sorted population and the global best solution
        """
        sorted_pop = self.get_sorted_population(pop, self.problem.minmax)
        c_best, c_worst = sorted_pop[0], sorted_pop[-1]
        if save:
            ## Save current best
            self.history.list_current_best.append(c_best)
            better = self.get_better_agent(c_best, self.history.list_global_best[-1], self.problem.minmax)
            self.history.list_global_best.append(better)
            ## Save current worst
            self.history.list_current_worst.append(c_worst)
            worse = self.get_better_agent(c_worst, self.history.list_global_worst[-1], self.problem.minmax, reverse=True)
            self.history.list_global_worst.append(worse)
            return sorted_pop, better
        else:
            ## Handle current best
            local_better = self.get_better_agent(c_best, self.history.list_current_best[-1], self.problem.minmax)
            self.history.list_current_best[-1] = local_better
            global_better = self.get_better_agent(c_best, self.history.list_global_best[-1], self.problem.minmax)
            self.history.list_global_best[-1] = global_better
            ## Handle current worst
            local_worst = self.get_better_agent(c_worst, self.history.list_current_worst[-1], self.problem.minmax, reverse=True)
            self.history.list_current_worst[-1] = local_worst
            global_worst = self.get_better_agent(c_worst, self.history.list_global_worst[-1], self.problem.minmax, reverse=True)
            self.history.list_global_worst[-1] = global_worst
            return sorted_pop, global_better

    ## Selection techniques
    def get_index_roulette_wheel_selection(self, list_fitness: np.array):
        """
        This method can handle min/max problem, and negative or positive fitness value.

        Args:
            list_fitness (nd.array): 1-D numpy array

        Returns:
            int: Index of selected solution
        """
        if type(list_fitness) in [list, tuple, np.ndarray]:
            list_fitness = np.array(list_fitness).flatten()
        if list_fitness.ptp() == 0:
            return int(self.generator.integers(0, len(list_fitness)))
        if np.any(list_fitness < 0):
            list_fitness = list_fitness - np.min(list_fitness)
        final_fitness = list_fitness
        if self.problem.minmax == "min":
            final_fitness = np.max(list_fitness) - list_fitness
        prob = final_fitness / np.sum(final_fitness)
        return int(self.generator.choice(range(0, len(list_fitness)), p=prob))

    def get_index_kway_tournament_selection(self, pop: List = None, k_way: float = 0.2, output: int = 2, reverse: bool = False) -> List:
        """
        Args:
            pop: The population
            k_way (float/int): The percent or number of solutions are randomized pick
            output (int): The number of outputs
            reverse (bool): set True when finding the worst fitness

        Returns:
            list: List of the selected indexes
        """
        if 0 < k_way < 1:
            k_way = int(k_way * len(pop))
        list_id = self.generator.choice(range(len(pop)), k_way, replace=False)
        list_parents = [[idx, pop[idx].target.fitness] for idx in list_id]
        if self.problem.minmax == "min":
            list_parents = sorted(list_parents, key=lambda agent: agent[1])
        else:
            list_parents = sorted(list_parents, key=lambda agent: agent[1], reverse=True)
        if reverse:
            return [parent[0] for parent in list_parents[-output:]]
        return [parent[0] for parent in list_parents[:output]]

    def get_levy_flight_step(self, beta: float = 1.0, multiplier: float = 0.001, 
                             size: Union[List, Tuple, np.ndarray] = None, case: int = 0) -> Union[float, List, np.ndarray]:
        """
        Get the Levy-flight step size

        Args:
            beta (float): Should be in range [0, 2].

                * 0-1: small range --> exploit
                * 1-2: large range --> explore

            multiplier (float): default = 0.001
            size (tuple, list): size of levy-flight steps, for example: (3, 2), 5, (4, )
            case (int): Should be one of these value [0, 1, -1].

                * 0: return multiplier * s * self.generator.uniform()
                * 1: return multiplier * s * self.generator.normal(0, 1)
                * -1: return multiplier * s

        Returns:
            float, list, np.ndarray: The step size of Levy-flight trajectory
        """
        # u and v are two random variables which follow self.generator.normal distribution
        # sigma_u : standard deviation of u
        sigma_u = np.power(gamma(1. + beta) * np.sin(np.pi * beta / 2) / (gamma((1 + beta) / 2.) * beta * np.power(2., (beta - 1) / 2)), 1. / beta)
        # sigma_v : standard deviation of v
        sigma_v = 1
        size = 1 if size is None else size
        u = self.generator.normal(0, sigma_u ** 2, size)
        v = self.generator.normal(0, sigma_v ** 2, size)
        s = u / np.power(np.abs(v), 1 / beta)
        if case == 0:
            step = multiplier * s * self.generator.uniform()
        elif case == 1:
            step = multiplier * s * self.generator.normal(0, 1)
        else:
            step = multiplier * s
        return step[0] if size == 1 else step

    def generate_opposition_solution(self, agent: Agent = None, g_best: Agent = None) -> np.ndarray:
        """
        Args:
            agent: The current agent
            g_best: the global best agent

        Returns:
            The opposite solution
        """
        pos_new = self.problem.lb + self.problem.ub - g_best.solution + self.generator.uniform() * (g_best.solution - agent.solution)
        return self.correct_solution(pos_new)

    def generate_group_population(self, pop: List[Agent], n_groups: int, m_agents: int) -> List:
        """
        Generate a list of group population from pop

        Args:
            pop: The current population
            n_groups: The n groups
            m_agents: The m agents in each group

        Returns:
            A list of group population
        """
        pop_group = []
        for idx in range(0, n_groups):
            group = pop[idx * m_agents: (idx + 1) * m_agents]
            pop_group.append([agent.copy() for agent in group])
        return pop_group

    ### Crossover
    def crossover_arithmetic(self, dad_pos=None, mom_pos=None):
        """
        Args:
            dad_pos: position of dad
            mom_pos: position of mom

        Returns:
            list: position of 1st and 2nd child
        """
        r = self.generator.uniform()  # w1 = w2 when r =0.5
        w1 = np.multiply(r, dad_pos) + np.multiply((1 - r), mom_pos)
        w2 = np.multiply(r, mom_pos) + np.multiply((1 - r), dad_pos)
        return w1, w2

    #### Improved techniques can be used in any algorithms: 1
    ## Based on this paper: An efficient equilibrium optimizer with mutation strategy for numerical optimization (but still different)
    ## This scheme used after the original and including 4 step:
    ##  s1: sort population, take p1 = 1/2 best population for next round
    ##  s2: do the mutation for p1, using greedy method to select the better solution
    ##  s3: do the search mechanism for p1 (based on global best solution and the updated p1 above), to make p2 population
    ##  s4: construct the new population for next generation
    def improved_ms(self, pop=None, g_best=None):  ## m: mutation, s: search
        pop_len = int(len(pop) / 2)
        ## Sort the updated population based on fitness
        pop = sorted(pop, key=lambda agent: agent.target.fitness)
        pop_s1, pop_s2 = pop[:pop_len], pop[pop_len:]

        ## Mutation scheme
        pop_new = []
        for idx in range(0, pop_len):
            agent = pop_s1[idx].copy()
            pos_new = pop_s1[idx].solution * (1 + self.generator.normal(0, 1, self.problem.n_dims))
            agent.solution = self.correct_solution(pos_new)
            pop_new.append(agent)
        pop_new = self.update_target_for_population(pop_new)
        pop_s1 = self.greedy_selection_population(pop_s1, pop_new, self.problem.minmax)  ## Greedy method --> improved exploitation

        ## Search Mechanism
        pos_s1_list = [agent.solution for agent in pop_s1]
        pos_s1_mean = np.mean(pos_s1_list, axis=0)
        pop_new = []
        for idx in range(0, pop_len):
            agent = pop_s2[idx].copy()
            pos_new = (g_best.solution - pos_s1_mean) - self.generator.random() * \
                      (self.problem.lb + self.generator.random() * (self.problem.ub - self.problem.lb))
            agent.solution = self.correct_solution(pos_new)
            pop_new.append(agent)
        ## Keep the diversity of populatoin and still improved the exploration
        pop_s2 = self.update_target_for_population(pop_new)
        pop_s2 = self.greedy_selection_population(pop_s2, pop_new, self.problem.minmax)

        ## Construct a new population
        pop = pop_s1 + pop_s2
        return pop

In [2]:
from math import pi
import numpy as np
import scipy as sp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.primitives import Estimator # new
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
import os
#from sko.SA import SA

In [3]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars = list(value_dict.values())
    expectation_value = estimator.run(ansatz, qubit_op, pars).result().values
    return np.real(expectation_value[0])  # Ensure it returns a scalar

In [4]:
class DevHS(Optimizer):
    def __init__(self, epoch: int = 10000, pop_size: int = 100, c_r: float = 0.95, pa_r: float = 0.05, **kwargs: object) -> None:
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.c_r = self.validator.check_float("c_r", c_r, (0, 1.0))
        self.pa_r = self.validator.check_float("pa_r", pa_r, (0, 1.0))
        self.set_parameters(["epoch", "pop_size", "c_r", "pa_r"])
        self.sort_flag = False
        self.function_evaluations = 0  # Add the counter here

    def initialize_variables(self):
        self.fw = 0.0001 * (self.problem.ub - self.problem.lb)  # Fret Width (Bandwidth)
        self.fw_damp = 0.9995  # Fret Width Damp Ratio
        self.dyn_fw = self.fw

    def evolve(self, epoch):
        pop_new = []
        for idx in range(0, self.pop_size):
            pos_new = self.generator.uniform(self.problem.lb, self.problem.ub)
            delta = self.dyn_fw * self.generator.normal(self.problem.lb, self.problem.ub)
            pos_new = np.where(self.generator.random(self.problem.n_dims) < self.c_r, self.g_best.solution, pos_new)
            x_new = pos_new + delta
            pos_new = np.where(self.generator.random(self.problem.n_dims) < self.pa_r, x_new, pos_new)
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop_new.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                # Increment the function evaluation counter here
                pop_new[-1].target = self.get_target(pos_new)
                self.function_evaluations += 1
        pop_new = self.update_target_for_population(pop_new)
        self.dyn_fw = self.dyn_fw * self.fw_damp
        self.pop = self.get_sorted_and_trimmed_population(self.pop + pop_new, self.pop_size, minmax=self.problem.minmax)
        self.get_FEs(self.function_evaluations)

In [10]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar
    
    problem_dict = {
    "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(1000.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = DevHS(epoch=2000, pop_size=20, c_r = 0.95, pa_r = 0.05)
    g_best = model.solve(problem_dict, qubits= k)
    print(f"Fitness: {g_best.target.fitness}")
    print(f"Fitness: {model.g_best.target.fitness}")

2024/11/29 02:19:41 PM, INFO, __main__.DevHS: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/11/29 02:19:42 PM, INFO, __main__.DevHS: >>>Problem: P, Epoch: 1, Current best: -1.12421875, Global best: -1.12421875, Runtime: 0.21361 seconds


20


2024/11/29 02:19:52 PM, INFO, __main__.DevHS: Solving single objective optimization problem.


...

2024/11/29 02:26:17 PM, INFO, __main__.DevHS: >>>Problem: P, Epoch: 501, Current best: -7.80625, Global best: -7.80625, Runtime: 0.32545 seconds


10020


2024/11/29 02:26:34 PM, INFO, __main__.DevHS: >>>Problem: P, Epoch: 551, Current best: -7.80625, Global best: -7.80625, Runtime: 0.30155 seconds


11020


KeyboardInterrupt: 

In [49]:
import numpy as np
from mealpy import FloatVar

class DevSMA(Optimizer):

    def __init__(self, epoch: int = 10000, pop_size: int = 100, p_t: float = 0.03, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
            p_t (float): probability threshold (z in the paper), default = 0.03
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.p_t = self.validator.check_float("p_t", p_t, (0, 1.0))
        self.set_parameters(["epoch", "pop_size", "p_t"])
        self.sort_flag = True
        self.function_evaluations = 0  # Initialize function evaluation counter

    def initialize_variables(self):
        self.weights = np.zeros((self.pop_size, self.problem.n_dims))

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        # plus eps to avoid denominator zero
        ss = self.g_best.target.fitness - self.pop[-1].target.fitness + self.EPSILON
        # calculate the fitness weight of each slime mold
        for idx in range(0, self.pop_size):
            # Eq.(2.5)
            if idx <= int(self.pop_size / 2):
                self.weights[idx] = 1 + self.generator.uniform(0, 1, self.problem.n_dims) * \
                                    np.log10((self.g_best.target.fitness - self.pop[idx].target.fitness) / ss + 1)
            else:
                self.weights[idx] = 1 - self.generator.uniform(0, 1, self.problem.n_dims) * \
                                    np.log10((self.g_best.target.fitness - self.pop[idx].target.fitness) / ss + 1)
        a = np.arctanh(-(epoch / self.epoch) + 1)  # Eq.(2.4)
        b = 1 - epoch / self.epoch
        pop_new = []
        for idx in range(0, self.pop_size):
            # Update the Position of search agent
            if self.generator.uniform() < self.p_t:  # Eq.(2.7)
                pos_new = self.problem.generate_solution()
            else:
                p = np.tanh(np.abs(self.pop[idx].target.fitness - self.g_best.target.fitness))  # Eq.(2.2)
                vb = self.generator.uniform(-a, a, self.problem.n_dims)  # Eq.(2.3)
                vc = self.generator.uniform(-b, b, self.problem.n_dims)
                # two positions randomly selected from population, apply for the whole problem size instead of 1 variable
                id_a, id_b = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}), 2, replace=False)
                pos_1 = self.g_best.solution + vb * (self.weights[idx] * self.pop[id_a].solution - self.pop[id_b].solution)
                pos_2 = vc * self.pop[idx].solution
                condition = self.generator.random(self.problem.n_dims) < p
                pos_new = np.where(condition, pos_1, pos_2)
            # Check bound and re-calculate fitness after each individual move
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            agent.target = self.get_target(pos_new)  # Ensure the objective function is evaluated
            self.function_evaluations += 1  # Increment the function evaluation counter
            pop_new.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_new = self.update_target_for_population(pop_new)
            self.pop = self.greedy_selection_population(self.pop, pop_new, self.problem.minmax)
        self.get_FEs(self.function_evaluations)


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar
    
    problem = {
        "obj_func": evaluate_expectation,
        "bounds": FloatVar(lb=(-100., )*dimension, ub=(100., )*dimension),
        "minmax": "min",
        "log_to": None,
    }

    ## Run the algorithm
    model = DevSMA(epoch=300, pop_size=100, pr=0.03)
    g_best = model.solve(problem, qubits = k)
    print(f"Best solution: {g_best.solution}, Best fitness: {g_best.target.fitness}")

ansatz_num_parameters= 12
100
5100
10100
15100
20100
25100
Number of function evaluations:  29900 for qubits:  3
Best solution: [-39.57685866  -1.1330219    6.99645736  11.73244226  35.2577254
  35.87039684 -12.48888179   8.1150677  -35.28863185 -19.65684865
  45.7096972  -28.65497077], Best fitness: -1.875
ansatz_num_parameters= 16
100
5100
10100
15100
20100
25100
Number of function evaluations:  29900 for qubits:  4
Best solution: [  3.6883505   -0.46609358  -3.29015197   0.71596426  -1.02131395
   0.16342431   3.32802648 -58.00974519 -53.04748245 -48.64273561
   4.84605314   4.76536084   0.81061796   1.60222097  95.58707334
  -0.1306963 ], Best fitness: -2.84375
ansatz_num_parameters= 20
100
5100
10100
15100
20100
25100
Number of function evaluations:  29900 for qubits:  5
Best solution: [-2.64888050e+00  1.20647580e+01 -3.98888426e+00  6.99183884e+01
 -9.25339713e-01 -7.63559720e-01 -1.53915122e-01 -1.71366251e+01
  2.80559483e-01 -7.53355432e-01 -2.64352431e-04 -5.19091525e+01
 -4

In [53]:
import numpy as np
from mealpy import FloatVar

class OriginalBBO(Optimizer):
    """
    The original version of: Biogeography-Based Optimization (BBO)

    Links:
        1. https://ieeexplore.ieee.org/abstract/document/4475427

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + p_m (float): (0, 1) -> better [0.01, 0.2], Mutation probability
        + n_elites (int): (2, pop_size/2) -> better [2, 5], Number of elites will be keep for next generation

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, BBO
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = BBO.OriginalBBO(epoch=1000, pop_size=50, p_m=0.01, n_elites=2)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")
    >>> print(f"Function evaluations: {model.get_function_evaluations()}")

    References
    ~~~~~~~~~~
    [1] Simon, D., 2008. Biogeography-based optimization. IEEE transactions on evolutionary computation, 12(6), pp.702-713.
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, p_m: float = 0.01, n_elites: int = 2, **kwargs: object) -> None:
        """
        Initialize the algorithm components.

        Args:
            epoch: Maximum number of iterations, default = 10000
            pop_size: Number of population size, default = 100
            p_m: Mutation probability, default=0.01
            n_elites: Number of elites will be keep for next generation, default=2
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.p_m = self.validator.check_float("p_m", p_m, (0., 1.0))
        self.n_elites = self.validator.check_int("n_elites", n_elites, [2, int(self.pop_size / 2)])
        self.set_parameters(["epoch", "pop_size", "p_m", "n_elites"])
        self.sort_flag = False
        self.mu = (self.pop_size + 1 - np.array(range(1, self.pop_size + 1))) / (self.pop_size + 1)
        self.mr = 1 - self.mu
        self.function_evaluations = 0  # Initialize function evaluation counter

    def evolve(self, epoch: int) -> None:
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch: The current iteration
        """
        _, pop_elites, _ = self.get_special_agents(self.pop, n_best=self.n_elites, minmax=self.problem.minmax)
        pop = []
        for idx in range(0, self.pop_size):
            # Probabilistic migration to the i-th position
            pos_new = self.pop[idx].solution.copy()
            for j in range(self.problem.n_dims):
                if self.generator.random() < self.mr[idx]:  # Should we immigrate?
                    # Pick a position from which to emigrate (roulette wheel selection)
                    random_number = self.generator.random() * np.sum(self.mu)
                    select = self.mu[0]
                    select_index = 0
                    while (random_number > select) and (select_index < self.pop_size - 1):
                        select_index += 1
                        select += self.mu[select_index]
                    # this is the migration step
                    pos_new[j] = self.pop[select_index].solution[j]
            noise = self.generator.uniform(self.problem.lb, self.problem.ub)
            condition = self.generator.random(self.problem.n_dims) < self.p_m
            pos_new = np.where(condition, noise, pos_new)
            pos_new = self.correct_solution(pos_new)
            agent_new = self.generate_empty_agent(pos_new)
            pop.append(agent_new)
            if self.mode not in self.AVAILABLE_MODES:
                agent_new.target = self.get_target(pos_new)
                self.function_evaluations += 1  # Increment function evaluation counter
                self.pop[idx] = self.get_better_agent(self.pop[idx], agent_new, self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop = self.update_target_for_population(pop)
            self.pop = self.greedy_selection_population(self.pop, pop, self.problem.minmax)
        # replace the solutions with their new migrated and mutated versions then Merge Populations
        self.pop = self.get_sorted_and_trimmed_population(self.pop + pop_elites, self.pop_size, self.problem.minmax)
        self.get_FEs(self.function_evaluations)  # Pass function evaluations to parent class


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar
    
    problem_dict = {
     "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(1000.,) * dimension, name="delta"),
     "minmax": "min",
     "obj_func": evaluate_expectation
    }

    model = OriginalBBO(epoch=1000, pop_size=100, p_m=0.01, n_elites=2)
    g_best = model.solve(problem_dict, qubits= k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 01:49:29 PM, INFO, __main__.OriginalBBO: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 01:49:30 PM, INFO, __main__.OriginalBBO: >>>Problem: P, Epoch: 1, Current best: -1.3125, Global best: -1.3125, Runtime: 0.38550 seconds


100


2024/07/12 01:49:39 PM, INFO, __main__.OriginalBBO: Solving single objective optimization problem.


...

2024/07/12 01:56:27 PM, INFO, __main__.OriginalBBO: >>>Problem: P, Epoch: 101, Current best: -5.0, Global best: -5.0, Runtime: 0.46158 seconds


10100


2024/07/12 01:56:49 PM, INFO, __main__.OriginalBBO: >>>Problem: P, Epoch: 151, Current best: -5.0, Global best: -5.0, Runtime: 0.60596 seconds


15100


KeyboardInterrupt: 

In [59]:
import numpy as np
from mealpy import FloatVar

class DevSOA(Optimizer):
    """
    The developed version: Seagull Optimization Algorithm (SOA)

    Links:
        1. https://www.sciencedirect.com/science/article/abs/pii/S0950705118305768

    Notes:
        1. The original one will not work because their operators always make the solution out of bound.
        2. I added the normal random number in Eq. 14 to make its work
        3. Besides, I will check keep the better one and remove the worst

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + fc (float): [1.0, 10.0] -> better [1, 5], freequency of employing variable A (A linear decreased from fc to 0), default = 2

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, SOA
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = SOA.DevSOA(epoch=1000, pop_size=50, fc = 2)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")
    >>> print(f"Function evaluations: {model.get_function_evaluations()}")
    """

    def __init__(self, epoch=10000, pop_size=100, fc=2, **kwargs):
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.fc = self.validator.check_float("fc", fc, [1.0, 10.])
        self.set_parameters(["epoch", "pop_size", "fc"])
        self.sort_flag = False
        self.function_evaluations = 0  # Initialize function evaluation counter

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        A = self.fc - epoch * self.fc / self.epoch    # Eq. 6
        uu = vv = 1
        pop_new = []
        for idx in range(0, self.pop_size):
            B = 2 * A**2 * self.generator.random()                          # Eq. 8
            M = B * (self.g_best.solution - self.pop[idx].solution)         # Eq. 7
            C = A * self.pop[idx].solution                                  # Eq. 5
            D = np.abs(C + M)                                               # Eq. 9
            k = self.generator.uniform(0, 2 * np.pi)
            r = uu * np.exp(k * vv)
            xx = r * np.cos(k)
            yy = r * np.sin(k)
            zz = r * k
            pos_new = xx * yy * zz * D + self.generator.normal(0, 1) * self.g_best.solution  # Eq. 14
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop_new.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.function_evaluations += 1  # Increment function evaluation counter
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_new = self.update_target_for_population(pop_new)
            self.pop = self.greedy_selection_population(self.pop, pop_new, self.problem.minmax)
        self.get_FEs(self.function_evaluations)  # Pass function evaluations to parent class


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
        "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
        "minmax": "min",
        "obj_func": evaluate_expectation
    }

    model = DevSOA(epoch=1000, pop_size=75, fc = 4)
    g_best = model.solve(problem_dict, qubits=k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 02:15:01 PM, INFO, __main__.DevSOA: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 02:15:02 PM, INFO, __main__.DevSOA: >>>Problem: P, Epoch: 1, Current best: -1.0625, Global best: -1.0625, Runtime: 0.23784 seconds


75


2024/07/12 02:15:13 PM, INFO, __main__.DevSOA: >>>Problem: P, Epoch: 51, Current best: -1.84375, Global best: -1.84375, Runtime: 0.19310 seconds


...

2024/07/12 02:15:14 PM, INFO, __main__.DevSOA: >>>Problem: P, Epoch: 1, Current best: -1.28125, Global best: -1.28125, Runtime: 0.21601 seconds


75


2024/07/12 02:15:26 PM, INFO, __main__.DevSOA: >>>Problem: P, Epoch: 51, Current best: -1.5, Global best: -1.5, Runtime: 0.21291 seconds


3825


KeyboardInterrupt: 

In [11]:
import numpy as np
from mealpy import FloatVar

class OriginalSOS(Optimizer):
    """
    The original version: Symbiotic Organisms Search (SOS)

    Links:
        1. https://doi.org/10.1016/j.compstruc.2014.03.007

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, SOS
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = SOS.OriginalSOS(epoch=1000, pop_size=50)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")
    >>> print(f"Function evaluations: {model.get_function_evaluations()}")
    """

    def __init__(self, epoch=10000, pop_size=100, **kwargs):
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.set_parameters(["epoch", "pop_size"])
        self.is_parallelizable = False
        self.sort_flag = False
        self.function_evaluations = 0  # Initialize function evaluation counter

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        for idx in range(0, self.pop_size):
            ## Mutualism Phase
            jdx = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}))
            mutual_vector = (self.pop[idx].solution + self.pop[jdx].solution) / 2
            bf1, bf2 = self.generator.integers(1, 3, 2)
            xi_new = self.pop[idx].solution + self.generator.random() * (self.g_best.solution - bf1 * mutual_vector)
            xj_new = self.pop[jdx].solution + self.generator.random() * (self.g_best.solution - bf2 * mutual_vector)
            xi_new = self.correct_solution(xi_new)
            xj_new = self.correct_solution(xj_new)
            xi_target = self.get_target(xi_new)
            self.function_evaluations += 1  # Increment function evaluation counter
            xj_target = self.get_target(xj_new)
            self.function_evaluations += 1  # Increment function evaluation counter
            if self.compare_target(xi_target, self.pop[idx].target, self.problem.minmax):
                self.pop[idx].update(solution=xi_new, target=xi_target)
            if self.compare_target(xj_target, self.pop[jdx].target, self.problem.minmax):
                self.pop[jdx].update(solution=xj_new, target=xj_target)
            ## Commensalism phase
            jdx = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}))
            xi_new = self.pop[idx].solution + self.generator.uniform(-1, 1) * (self.g_best.solution - self.pop[jdx].solution)
            xi_new = self.correct_solution(xi_new)
            xi_target = self.get_target(xi_new)
            self.function_evaluations += 1  # Increment function evaluation counter
            if self.compare_target(xi_target, self.pop[idx].target, self.problem.minmax):
                self.pop[idx].update(solution=xi_new, target=xi_target)
            ## Parasitism phase
            jdx = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}))
            temp_idx = self.generator.integers(0, self.problem.n_dims)
            xi_new = self.pop[jdx].solution.copy()
            xi_new[temp_idx] = self.problem.generate_solution()[temp_idx]
            xi_new = self.correct_solution(xi_new)
            xi_target = self.get_target(xi_new)
            self.function_evaluations += 1  # Increment function evaluation counter
            if self.compare_target(xi_target, self.pop[idx].target, self.problem.minmax):
                self.pop[idx].update(solution=xi_new, target=xi_target)
        self.get_FEs(self.function_evaluations)  # Pass function evaluations to parent class


In [12]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
        "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
        "minmax": "min",
        "obj_func": evaluate_expectation
    }

    model = OriginalSOS(epoch=1000, pop_size=20)
    g_best = model.solve(problem_dict, qubits = k)
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/11/29 02:30:49 PM, INFO, __main__.OriginalSOS: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/11/29 02:30:51 PM, INFO, __main__.OriginalSOS: >>>Problem: P, Epoch: 1, Current best: -1.325, Global best: -1.325, Runtime: 0.99778 seconds


80


2024/11/29 02:31:07 PM, INFO, __main__.OriginalSOS: Solving single objective optimization problem.


...

2024/11/29 02:41:17 PM, INFO, __main__.OriginalSOS: >>>Problem: P, Epoch: 251, Current best: -6.817187499999999, Global best: -6.817187499999999, Runtime: 0.91357 seconds


20080


2024/11/29 02:42:02 PM, INFO, __main__.OriginalSOS: >>>Problem: P, Epoch: 301, Current best: -6.8804687499999995, Global best: -6.8804687499999995, Runtime: 0.96704 seconds


24080


KeyboardInterrupt: 

In [79]:
import numpy as np
from mealpy import FloatVar

class DevTPO(Optimizer):
    """
    The original version: Tree Physiology Optimization (TPO)

    Links:
        1. https://www.mathworks.com/matlabcentral/fileexchange/63982-tree-physiology-optimization-tpo-algorithm-for-stochastic-test-function-optimization

    Notes:
        1. The paper is difficult to read and understand, and the provided MATLAB code is also challenging to understand.
        2. Based on my idea:
            + pop_size = number of branhes, the population size should be equal to the number of branches.
            + The number of leaves should be calculated as int(sqrt(pop_size) + 1), so we don't need to specify the n_leafs parameter, which will also reduce computation time.
            + When using this algorithm, especially when setting stopping conditions, be careful and set it to the FE type.

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + alpha (float): [-10, 10.] -> better [0.2, 0.5], Absorption constant for tree root elongation, default = 0.5
        + beta (float): [-100, 100.] -> better [10, 50], Diversification facor of tree shoot, default=50.
        + theta (float): (0, 1.0] -> better [0.5, 0.9], Factor to reduce randomization, Theta = Power law to reduce randomization as iteration increases, default=0.9

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, TPO
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = TPO.DevTPO(epoch=1000, pop_size=50, alpha = 0.3, beta = 50., theta = 0.9)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")
    >>> print(f"Function evaluations: {model.get_function_evaluations()}")
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, alpha: float = 0.3, beta: float = 50.0, theta: float = 0.9, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
            alpha (float): Absorption constant for tree root elongation, default=0.3
            beta (float): Diversification factor of tree shoot, default=50.
            theta (float): Factor to reduce randomization, Theta = Power law to reduce randomization as iteration increases, default=0.9
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])     # Number of branches
        self.alpha = self.validator.check_float("alpha", alpha, [-10.0, 10.])
        self.beta = self.validator.check_float("beta", beta, [-100., 100])
        self.theta = self.validator.check_float("theta", theta, (0, 1.0))
        self.set_parameters(["epoch", "pop_size", "alpha", "beta", "theta"])
        self.sort_flag = False
        self.function_evaluations = 0  # Initialize function evaluation counter

    def initialize_variables(self):
        """
        The idea is a tree has a pop_size of branches (n_branches), each branch will have several leafs.
        """
        self.n_leafs = int(np.sqrt(self.pop_size) + 1)  # Number of leafs
        self._theta = self.theta
        self.roots = self.generator.uniform(0, 1, (self.n_leafs, self.problem.n_dims))

    def initialization(self):
        self.pop_total = []
        self.pop = []                       # The best leaf in each branches
        for idx in range(self.pop_size):
            leafs = self.generate_population(self.n_leafs)
            best = self.get_best_agent(leafs, self.problem.minmax)
            self.pop.append(best)
            self.pop_total.append(leafs)
            self.function_evaluations += self.n_leafs  # Increment function evaluation counter

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        for idx in range(0, self.pop_size):
            pos_list = np.array([agent.solution for agent in self.pop_total[idx]])
            carbon_gain = self._theta * self.g_best.solution - pos_list
            roots_old = np.copy(self.roots)
            self.roots += self.alpha * carbon_gain * self.generator.uniform(-0.5, 0.5, (self.n_leafs, self.problem.n_dims))
            nutrient_value = self._theta * (self.roots - roots_old)
            pos_list_new = self.g_best.solution + self.beta * nutrient_value
            pop_new = []
            for jdx in range(0, self.n_leafs):
                pos_new = self.correct_solution(pos_list_new[jdx])
                agent = self.generate_empty_agent(pos_new)
                pop_new.append(agent)
                if self.mode not in self.AVAILABLE_MODES:
                    agent.target = self.get_target(pos_new)
                    self.function_evaluations += 1  # Increment function evaluation counter
                    self.pop_total[idx][jdx] = self.get_better_agent(agent, self.pop_total[idx][jdx], self.problem.minmax)
            if self.mode in self.AVAILABLE_MODES:
                pop_new = self.update_target_for_population(pop_new)
                self.pop_total[idx] = self.greedy_selection_population(pop_new, self.pop_total[idx], self.problem.minmax)
                self.function_evaluations += len(pop_new)  # Increment function evaluation counter
        self._theta = self._theta * self.theta
        for idx in range(0, self.pop_size):
            best = self.get_best_agent(self.pop_total[idx], self.problem.minmax)
            self.pop[idx] = best
        self.get_FEs(self.function_evaluations)


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = DevTPO(epoch=1000, pop_size=50, alpha = 0.3, beta = 50., theta = 0.9)
    g_best = model.solve(problem_dict, qubits = k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 03:31:03 PM, INFO, __main__.DevTPO: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 03:31:05 PM, INFO, __main__.DevTPO: >>>Problem: P, Epoch: 1, Current best: -1.65625, Global best: -1.65625, Runtime: 1.19650 seconds


800


2024/07/12 03:31:51 PM, INFO, __main__.DevTPO: Solving single objective optimization problem.


...

2024/07/12 03:40:17 PM, INFO, __main__.DevTPO: >>>Problem: P, Epoch: 251, Current best: -4.5, Global best: -4.5, Runtime: 1.20553 seconds


100800


2024/07/12 03:41:20 PM, INFO, __main__.DevTPO: >>>Problem: P, Epoch: 301, Current best: -4.5, Global best: -4.5, Runtime: 1.14705 seconds


120800


KeyboardInterrupt: 

In [82]:
import numpy as np
from mealpy import FloatVar

class DevVCS(Optimizer):
    """
    The developed version: Virus Colony Search (VCS)

    Links:
        1. https://doi.org/10.1016/j.advengsoft.2015.11.004

    Notes:
        + In Immune response process, updates the whole position instead of updating each variable in position

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + lamda (float): (0, 1.0) -> better [0.2, 0.5], Percentage of the number of the best will keep, default = 0.5
        + sigma (float): (0, 5.0) -> better [0.1, 2.0], Weight factor

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, VCS
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = VCS.DevVCS(epoch=1000, pop_size=50, lamda = 0.5, sigma = 0.3)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")
    >>> print(f"Function evaluations: {model.get_function_evaluations()}")
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, lamda: float = 0.5, sigma: float = 1.5, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
            lamda (float): Percentage of the number of the best will keep, default = 0.5
            sigma (float): Weight factor, default = 1.5
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.lamda = self.validator.check_float("lamda", lamda, (0, 1.0))
        self.sigma = self.validator.check_float("sigma", sigma, (0, 5.0))
        self.n_best = int(self.lamda * self.pop_size)
        self.set_parameters(["epoch", "pop_size", "lamda", "sigma"])
        self.sort_flag = True
        self.function_evaluations = 0  # Initialize function evaluation counter

    def calculate_xmean__(self, pop):
        """
        Calculate the mean position of list of solutions (population)

        Args:
            pop (list): List of solutions (population)

        Returns:
            list: Mean position
        """
        ## Calculate the weighted mean of the λ best individuals by
        pop = self.get_sorted_population(pop, self.problem.minmax)
        pos_list = [agent.solution for agent in pop[:self.n_best]]
        factor_down = self.n_best * np.log1p(self.n_best + 1) - np.log1p(np.prod(range(1, self.n_best + 1)))
        weight = np.log1p(self.n_best + 1) / factor_down
        weight = weight / self.n_best
        x_mean = weight * np.sum(pos_list, axis=0)
        return x_mean

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        ## Viruses diffusion
        pop = []
        for idx in range(0, self.pop_size):
            sigma = (np.log1p(epoch + 1) / self.epoch) * (self.pop[idx].solution - self.g_best.solution)
            gauss = self.generator.normal(self.generator.normal(self.g_best.solution, np.abs(sigma)))
            pos_new = gauss + self.generator.uniform() * self.g_best.solution - self.generator.uniform() * self.pop[idx].solution
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.function_evaluations += 1  # Increment function evaluation counter
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop = self.update_target_for_population(pop)
            self.function_evaluations += len(pop)  # Increment function evaluation counter
            self.pop = self.greedy_selection_population(self.pop, pop, self.problem.minmax)
        ## Host cells infection
        x_mean = self.calculate_xmean__(self.pop)
        sigma = self.sigma * (1 - epoch / self.epoch)
        pop = []
        for idx in range(0, self.pop_size):
            ## Basic / simple version, not the original version in the paper
            pos_new = x_mean + sigma * self.generator.normal(0, 1, self.problem.n_dims)
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.function_evaluations += 1  # Increment function evaluation counter
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop = self.update_target_for_population(pop)
            self.function_evaluations += len(pop)  # Increment function evaluation counter
            self.pop = self.greedy_selection_population(self.pop, pop, self.problem.minmax)
        ## Calculate the weighted mean of the λ best individuals by
        self.pop = self.get_sorted_population(self.pop, self.problem.minmax)
        ## Immune response
        pop = []
        for idx in range(0, self.pop_size):
            pr = (self.problem.n_dims - idx + 1) / self.problem.n_dims
            id1, id2 = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}), 2, replace=False)
            temp = self.pop[id1].solution - (self.pop[id2].solution - self.pop[idx].solution) * self.generator.uniform()
            condition = self.generator.random(self.problem.n_dims) < pr
            pos_new = np.where(condition, self.pop[idx].solution, temp)
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.function_evaluations += 1  # Increment function evaluation counter
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop = self.update_target_for_population(pop)
            self.function_evaluations += len(pop)  # Increment function evaluation counter
            self.pop = self.greedy_selection_population(self.pop, pop, self.problem.minmax)
        self.get_FEs(self.function_evaluations)



In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = DevVCS(epoch=1000, pop_size=50, lamda = 0.5, sigma = 0.3)
    g_best = model.solve(problem_dict, qubits = k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 03:46:27 PM, INFO, __main__.DevVCS: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 03:46:28 PM, INFO, __main__.DevVCS: >>>Problem: P, Epoch: 1, Current best: -1.28125, Global best: -1.28125, Runtime: 0.35455 seconds


150


2024/07/12 03:46:49 PM, INFO, __main__.DevVCS: >>>Problem: P, Epoch: 51, Current best: -1.8125, Global best: -1.8125, Runtime: 0.41727 seconds


...

2024/07/12 04:06:56 PM, INFO, __main__.DevVCS: >>>Problem: P, Epoch: 751, Current best: -5.5, Global best: -5.5, Runtime: 0.54347 seconds


112650


2024/07/12 04:07:25 PM, INFO, __main__.DevVCS: >>>Problem: P, Epoch: 801, Current best: -5.5, Global best: -5.5, Runtime: 0.54635 seconds


120150


KeyboardInterrupt: 

In [85]:
class OriginalFLA(Optimizer):
    """
    The original version of: Fick's Law Algorithm (FLA)

    Notes:
        1. The algorithm contains a high number of parameters, some of which may be unnecessary.
        2. Despite the complexity of the algorithms, they may not perform optimally and could potentially become trapped in local optima.
        3. Division by the fitness value may cause overflow issues to arise.
        4. https://www.mathworks.com/matlabcentral/fileexchange/121033-fick-s-law-algorithm-fla

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + C1 (float): factor C1, default=0.5
        + C2 (float): factor C2, default=2.0
        + C3 (float): factor C3, default=0.1
        + C4 (float): factor C4, default=0.2
        + C5 (float): factor C5, default=2.0
        + DD (float): factor D in the paper, default=0.01

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, FLA
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = FLA.OriginalFLA(epoch=1000, pop_size=50, C1 = 0.5, C2 = 2.0, C3 = 0.1, C4 = 0.2, C5 = 2.0, DD = 0.01)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

    References
    ~~~~~~~~~~
    [1] Hashim, F. A., Mostafa, R. R., Hussien, A. G., Mirjalili, S., & Sallam, K. M. (2023). Fick’s Law Algorithm: A physical
    law-based algorithm for numerical optimization. Knowledge-Based Systems, 260, 110146.
    """
    def __init__(self, epoch: int = 10000, pop_size: int = 100, C1: float = 0.5, C2: float = 2.0,
                 C3: float = 0.1, C4: float = 0.2, C5: float = 2.0, DD: float = 0.01, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
            C1 (float): factor C1, default=0.5
            C2 (float): factor C2, default=2.0
            C3 (float): factor C3, default=0.1
            C4 (float): factor C4, default=0.2
            C5 (float): factor C5, default=2.0
            DD (float): factor D in the paper, default=0.01
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [10, 10000])
        self.C1 = self.validator.check_float("C1", C1, (-100., 100.))
        self.C2 = self.validator.check_float("C2", C2, (-100., 100.))
        self.C3 = self.validator.check_float("C3", C3, (-100., 100.))
        self.C4 = self.validator.check_float("C4", C4, (-100., 100.))
        self.C5 = self.validator.check_float("C5", C5, (-100., 100.))
        self.DD = self.validator.check_float("DD", DD, (-100., 100.))
        self.set_parameters(["epoch", "pop_size", "C1", "C2", "C3", "C4", "C5", "DD"])
        self.sort_flag = False
        self.function_evaluations = 0  # Initialize the function evaluations counter

    def count_fe(self):
        """Increment function evaluation counter."""
        self.function_evaluations += 1

    def before_main_loop(self):
        self.xss = self.get_sorted_population(self.pop, self.problem.minmax)
        self.g_best = self.xss[0].copy()
        self.n1 = int(np.round(self.pop_size/2))
        self.n2 = self.pop_size - self.n1
        self.pop1 = self.pop[:self.n1].copy()
        self.pop2 = self.pop[self.n1:].copy()
        self.best1 = self.get_best_agent(self.pop1, self.problem.minmax)
        self.best2 = self.get_best_agent(self.pop2, self.problem.minmax)
        if self.compare_target(self.best1.target, self.best2.target, self.problem.minmax):
            self.fsss = self.best1.target.fitness
        else:
            self.fsss = self.best2.target.fitness

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        pos_list = np.array([agent.solution for agent in self.pop])
        pos1_list = np.array([agent.solution for agent in self.pop1])
        pos2_list = np.array([agent.solution for agent in self.pop2])
        xm1 = np.mean(pos1_list, axis=0)
        xm2 = np.mean(pos2_list, axis=0)
        xm = np.mean(pos_list, axis=0)
        tf = np.sinh((epoch+1) / self.epoch)**self.C1
        pop_new = []
        if tf < 0.9:
            dof = np.exp(-(self.C2 * tf - self.generator.random()))**self.C2
            tdo = self.C5 * tf - self.generator.random()
            if tdo < self.generator.random():
                m1n, m2n = self.C3*self.n1, self.C4*self.n1
                nt12 = int(np.round((m2n - m1n)*self.generator.random() + m1n))
                for idx in range(0, nt12):
                    dfg = self.generator.integers(1, 3)
                    jj = -self.DD * (xm2 - xm1) / np.linalg.norm(self.best2.solution - self.pop1[idx].solution + self.EPSILON)
                    pos_new = self.best2.solution + dfg*dof*self.generator.random(self.problem.n_dims)*(jj*self.best2.solution - self.pop1[idx].solution)
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
                for idx in range(nt12, self.n1):
                    tt = self.pop1[idx].solution + dof * (self.generator.random(self.problem.n_dims) * (self.problem.ub - self.problem.lb) + self.problem.lb)
                    pp = self.generator.random(self.problem.n_dims)
                    pos_new = np.where(pp < 0.8, self.best1.solution, np.where(pp >= 0.9, self.pop1[idx].solution, tt))
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
                for idx in range(0, self.n2):
                    pos_new = self.best2.solution + dof * (self.generator.random(self.problem.n_dims) * (self.problem.ub - self.problem.lb) + self.problem.lb)
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
            else:
                m1n, m2n = 0.1 * self.n2, 0.2 * self.n2
                nt12 = int(np.round((m2n - m1n) * self.generator.random() + m1n))
                for idx in range(0, nt12):
                    dfg = self.generator.integers(1, 3)
                    jj = -self.DD*(xm1-xm2) / np.linalg.norm(self.best1.solution - self.pop2[idx].solution + self.EPSILON)
                    pos_new = self.best1.solution + dfg * dof * self.generator.random(self.problem.n_dims) * (jj * self.best1.solution - self.pop2[idx].solution)
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
                for idx in range(nt12, self.n2):
                    tt = self.pop2[idx].solution + dof * (self.generator.random(self.problem.n_dims) * (self.problem.ub - self.problem.lb) + self.problem.lb)
                    pp = self.generator.random(self.problem.n_dims)
                    pos_new = np.where(pp < 0.8, self.best2.solution, np.where(pp >= 0.9, self.pop2[idx].solution, tt))
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
                for idx in range(0, self.n1):
                    pos_new = self.best1.solution + dof * (self.generator.random(self.problem.n_dims) * (self.problem.ub - self.problem.lb) + self.problem.lb)
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
        else:       # Equilibrium operator (EO)
            if tf <= 1:
                for idx in range(0, self.n1):
                    dfg = self.generator.integers(1, 3)
                    tttt = np.linalg.norm(self.best1.solution - self.pop1[idx].solution)
                    if tttt == 0:
                        jj = 0
                    else:
                        jj = -self.DD*(self.best1.solution - xm1) / tttt
                    drf = np.exp(-jj / tf)
                    ms = np.exp(-self.best1.target.fitness / self.pop1[idx].target.fitness + self.EPSILON)
                    qeo = dfg * drf * self.generator.random(self.problem.n_dims)
                    pos_new = self.best1.solution + qeo*self.pop1[idx].solution + qeo *(ms * self.best1.solution - self.pop1[idx].solution)
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
                for idx in range(0, self.n2):
                    dfg = self.generator.integers(1, 3)
                    tttt = np.linalg.norm(self.best2.solution - self.pop2[idx].solution)
                    if tttt == 0:
                        jj = 0
                    else:
                        jj = -self.DD * (self.best2.solution - xm2) / tttt
                    drf = np.exp(-jj / tf)
                    ms = np.exp(-self.best2.target.fitness / self.pop2[idx].target.fitness + self.EPSILON)
                    qeo = dfg * drf * self.generator.random(self.problem.n_dims)
                    pos_new = self.best2.solution + qeo * self.pop2[idx].solution + qeo * (ms * self.best2.solution - self.pop2[idx].solution)
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
            else:   # Steady state operator (SSO)
                for idx in range(0, self.n1):
                    dfg = self.generator.integers(1, 3)
                    tttt = np.linalg.norm(self.g_best.solution - self.pop1[idx].solution)
                    if tttt == 0:
                        jj = 0
                    else:
                        jj = -self.DD * (xm - xm1) / tttt
                    drf = np.exp(-jj / tf)
                    ms = np.exp(-self.fsss / self.pop1[idx].target.fitness + self.EPSILON)
                    qg = dfg * drf * self.generator.random(self.problem.n_dims)
                    pos_new = self.g_best.solution + qg * self.pop1[idx].solution + qg * (ms * self.best1.solution - self.pop1[idx].solution)
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
                for idx in range(0, self.n2):
                    dfg = self.generator.integers(1, 3)
                    tttt = np.linalg.norm(self.g_best.solution - self.pop2[idx].solution)
                    if tttt == 0:
                        jj = 0
                    else:
                        jj = -self.DD * (xm - xm2) / tttt
                    drf = np.exp(-jj / tf)
                    ms = np.exp(-self.fsss / self.pop2[idx].target.fitness + self.EPSILON)
                    qg = dfg * drf * self.generator.random(self.problem.n_dims)
                    pos_new = self.g_best.solution + qg * self.pop2[idx].solution + qg * (ms * self.g_best.solution - self.pop2[idx].solution)
                    pos_new = self.correct_solution(pos_new)
                    agent = self.generate_empty_agent(pos_new)
                    pop_new.append(agent)
                    self.count_fe()  # Increment function evaluation counter
        if self.mode not in self.AVAILABLE_MODES:
            for idx in range(0, self.pop_size):
                pop_new[idx].target = self.get_target(pop_new[idx].solution)
        else:
            pop_new = self.update_target_for_population(pop_new)
        for idx in range(0, self.pop_size):
            if self.compare_target(pop_new[idx].target, self.pop[idx].target, self.problem.minmax):
                self.pop[idx] = pop_new[idx]
        self.pop1 = self.pop[:self.n1].copy()
        self.pop2 = self.pop[self.n1:].copy()
        self.best1 = self.get_best_agent(self.pop1, self.problem.minmax)
        self.best2 = self.get_best_agent(self.pop2, self.problem.minmax)
        if self.compare_target(self.best1.target, self.best2.target, self.problem.minmax):
            self.fsss = self.best1.target.fitness
        else:
            self.fsss = self.best2.target.fitness
        self.get_FEs(self.function_evaluations)


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                backend_options={ # method chosen automatically to match options
                    "coupling_map": coupling_map,
                },
                run_options={"seed": 42, "shots": 1024},
                transpile_options={"seed_transpiler": 42},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = OriginalFLA(epoch=1000, pop_size=50, C1 = 0.5, C2 = 2.0, C3 = 0.1, C4 = 0.2, C5 = 2.0, DD = 0.01)
    g_best = model.solve(problem_dict, qubits= k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 04:17:25 PM, INFO, __main__.OriginalFLA: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 04:17:25 PM, INFO, __main__.OriginalFLA: >>>Problem: P, Epoch: 1, Current best: -1.5, Global best: -1.5, Runtime: 0.17218 seconds


50


2024/07/12 04:17:25 PM, INFO, __main__.OriginalFLA: Solving single objective optimization problem.


...

2024/07/12 04:31:36 PM, INFO, __main__.OriginalFLA: Solving single objective optimization problem.


Number of function evaluations:  49950 for qubits:  8
Solution: [  -5.56206149   81.62130134  -56.60883797  100.           92.24275055
  -53.71685116  -36.44606684  -91.26000198  -42.11249224   62.0895334
   95.6718241    72.03873556  -45.80980187  -29.2037962    58.58025721
 -100.           41.04127075  -33.12709315   67.69774598  -20.28041149
   55.15787845   79.85523559   72.82069777  -60.88403154 -100.
   42.75176977  100.           11.06255625    9.93499379   20.25976189
   19.79595718   17.82468347], Fitness: -6.21875
Solution: [  -5.56206149   81.62130134  -56.60883797  100.           92.24275055
  -53.71685116  -36.44606684  -91.26000198  -42.11249224   62.0895334
   95.6718241    72.03873556  -45.80980187  -29.2037962    58.58025721
 -100.           41.04127075  -33.12709315   67.69774598  -20.28041149
   55.15787845   79.85523559   72.82069777  -60.88403154 -100.
   42.75176977  100.           11.06255625    9.93499379   20.25976189
   19.79595718   17.82468347], Fitness: -6.

2024/07/12 04:31:36 PM, INFO, __main__.OriginalFLA: >>>Problem: P, Epoch: 1, Current best: -2.46875, Global best: -2.46875, Runtime: 0.24711 seconds


50


KeyboardInterrupt: 

In [95]:
import numpy as np
import math
from mealpy import FloatVar

class OriginalNRO(Optimizer):
    """
    The original version of: Nuclear Reaction Optimization (NRO)

    Links:
        1. https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=8720256

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, MVO
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = MVO.DevMVO(epoch=1000, pop_size=50)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

    References
    ~~~~~~~~~~
    [1] Wei, Z., Huang, C., Wang, X., Han, T. and Li, Y., 2019. Nuclear reaction optimization: A novel and
    powerful physics-based algorithm for global optimization. IEEE Access, 7, pp.66084-66109.
    [2] Wei, Z.L., Zhang, Z.R., Huang, C.Q., Han, B., Tang, S.Q. and Wang, L., 2019, June. An Approach
    Inspired from Nuclear Reaction Processes for Numerical Optimization. In Journal of Physics:
    Conference Series (Vol. 1213, No. 3, p. 032009). IOP Publishing.
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.set_parameters(["epoch", "pop_size"])
        self.sort_flag = False
        self.function_evaluations = 0  # Initialize function evaluations counter

    def amend_solution(self, solution: np.ndarray) -> np.ndarray:
        rand_pos = self.generator.uniform(self.problem.lb, self.problem.ub)
        condition = np.logical_and(self.problem.lb <= solution, solution <= self.problem.ub)
        return np.where(condition, solution, rand_pos)

    def count_fe(self):
        """Increment function evaluation counter"""
        self.function_evaluations += 1

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class
        """
        xichma_v = 1
        xichma_u = ((math.gamma(1 + 1.5) * math.sin(math.pi * 1.5 / 2)) /
                    (math.gamma((1 + 1.5) / 2) * 1.5 * 2 ** ((1.5 - 1) / 2))) ** (1.0 / 1.5)
        levy_b = (self.generator.normal(0, xichma_u ** 2)) / (
                    np.sqrt(np.abs(self.generator.normal(0, xichma_v ** 2))) ** (1.0 / 1.5))
        # NFi phase
        Pb = self.generator.uniform()
        Pfi = self.generator.uniform()
        freq = 0.05
        alpha = 0.01
        pop_new = []
        for idx in range(self.pop_size):
            ## Calculate neutron vector Nei by Eq. (2)
            ## Random 1 more index to select neutron
            temp1 = list(set(range(0, self.pop_size)) - {idx})
            i1 = self.generator.choice(temp1, replace=False)
            Nei = (self.pop[idx].solution + self.pop[i1].solution) / 2
            ## Update population of fission products according to Eq.(3), (6) or (9);
            if self.generator.uniform() <= Pfi:
                ### Update based on Eq. 3
                if self.generator.uniform() <= Pb:
                    xichma1 = (np.log(epoch) * 1.0 / epoch) * np.abs(
                        np.subtract(self.pop[idx].solution, self.g_best.solution))
                    gauss = np.array(
                        [self.generator.normal(self.g_best.solution[j], xichma1[j]) for j in
                         range(self.problem.n_dims)])
                    Xi = gauss + self.generator.uniform() * self.g_best.solution - round(
                        self.generator.random() + 1) * Nei
                ### Update based on Eq. 6
                else:
                    i2 = self.generator.choice(temp1, replace=False)
                    xichma2 = (np.log(epoch) * 1.0 / epoch) * np.abs(
                        np.subtract(self.pop[i2].solution, self.g_best.solution))
                    gauss = np.array(
                        [self.generator.normal(self.pop[idx].solution[j], xichma2[j]) for j in
                         range(self.problem.n_dims)])
                    Xi = gauss + self.generator.uniform() * self.g_best.solution - round(
                        self.generator.random() + 2) * Nei
            ## Update based on Eq. 9
            else:
                i3 = self.generator.choice(temp1, replace=False)
                xichma2 = (np.log(epoch) * 1.0 / epoch) * np.abs(
                    np.subtract(self.pop[i3].solution, self.g_best.solution))
                Xi = np.array(
                    [self.generator.normal(self.pop[idx].solution[j], xichma2[j]) for j in
                     range(self.problem.n_dims)])
            ## Check the boundary and evaluate the fitness function
            pos_new = self.correct_solution(Xi)
            agent = self.generate_empty_agent(pos_new)
            pop_new.append(agent)
            self.count_fe()  # Increment function evaluation counter
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_new = self.update_target_for_population(pop_new)
            self.pop = self.greedy_selection_population(self.pop, pop_new, self.problem.minmax)

        # NFu phase
        ## Ionization stage
        ## Calculate the Pa through Eq. (10)
        pop_child = []
        ranked_pop = np.argsort([self.pop[i].target.fitness for i in range(self.pop_size)])
        for idx in range(self.pop_size):
            X_ion = self.pop[idx].solution.copy()
            if (ranked_pop[idx] * 1.0 / self.pop_size) < self.generator.random():
                i1, i2 = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}), 2, replace=False)
                for j in range(self.problem.n_dims):
                    #### Levy flight strategy is described as Eq. 18
                    if self.pop[i2].solution[j] == self.pop[idx].solution[j]:
                        X_ion[j] = self.pop[idx].solution[j] + alpha * levy_b * (
                                    self.pop[idx].solution[j] - self.g_best.solution[j])
                    #### If not, based on Eq. 11, 12
                    else:
                        if self.generator.uniform() <= 0.5:
                            X_ion[j] = self.pop[i1].solution[j] + self.generator.uniform() * (
                                        self.pop[i2].solution[j] - self.pop[idx].solution[j])
                        else:
                            X_ion[j] = self.pop[i1].solution[j] - self.generator.uniform() * (
                                        self.pop[i2].solution[j] - self.pop[idx].solution[j])
            else:  #### Levy flight strategy is described as Eq. 21
                _, _, worst = self.get_special_agents(self.pop, n_worst=1, minmax=self.problem.minmax)
                X_worst = worst[0]
                for j in range(self.problem.n_dims):
                    ##### Based on Eq. 21
                    if X_worst.solution[j] == self.g_best.solution[j]:
                        X_ion[j] = self.pop[idx].solution[j] + alpha * levy_b * (
                                    self.problem.ub[j] - self.problem.lb[j])
                    ##### Based on Eq. 13
                    else:
                        X_ion[j] = self.pop[idx].solution[j] + round(self.generator.uniform()) * self.generator.uniform() * \
                                   (X_worst.solution[j] - self.g_best.solution[j])
            ## Check the boundary and evaluate the fitness function for X_ion
            pos_new = self.correct_solution(X_ion)
            agent = self.generate_empty_agent(pos_new)
            pop_child.append(agent)
            self.count_fe()  # Increment function evaluation counter
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_child = self.update_target_for_population(pop_child)
            self.pop = self.greedy_selection_population(self.pop, pop_child, self.problem.minmax)

        ## Fusion Stage
        ### all ions obtained from ionization are ranked based on (14) - Calculate the Pc through Eq. (14)
        pop_new = []
        ranked_pop = np.argsort([self.pop[i].target.fitness for i in range(self.pop_size)])
        for idx in range(self.pop_size):
            i1, i2 = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}), 2, replace=False)
            #### Generate fusion nucleus
            if (ranked_pop[idx] * 1.0 / self.pop_size) < self.generator.random():
                t1 = self.generator.uniform() * (self.pop[i1].solution - self.g_best.solution)
                t2 = self.generator.uniform() * (self.pop[i2].solution - self.g_best.solution)
                temp2 = self.pop[i1].solution - self.pop[i2].solution
                X_fu = self.pop[idx].solution + t1 + t2 - np.exp(-np.linalg.norm(temp2)) * temp2
            #### Else
            else:
                ##### Based on Eq. 22
                check_equal = (self.pop[i1].solution == self.pop[i2].solution)
                if check_equal.all():
                    X_fu = self.pop[idx].solution + alpha * levy_b * (self.pop[idx].solution - self.g_best.solution)
                ##### Based on Eq. 16, 17
                else:
                    if self.generator.uniform() > 0.5:
                        X_fu = self.pop[idx].solution - 0.5 * (np.sin(2 * np.pi * freq * epoch + np.pi) *
                            (self.epoch - epoch) / self.epoch + 1) * (self.pop[i1].solution - self.pop[i2].solution)
                    else:
                        X_fu = self.pop[idx].solution - 0.5 * (np.sin(2 * np.pi * freq * epoch + np.pi) * epoch / self.epoch + 1) * \
                               (self.pop[i1].solution - self.pop[i2].solution)
            pos_new = self.correct_solution(X_fu)
            agent = self.generate_empty_agent(pos_new)
            pop_new.append(agent)
            self.count_fe()  # Increment function evaluation counter
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_new = self.update_target_for_population(pop_new)
            self.pop = self.greedy_selection_population(self.pop, pop_new, self.problem.minmax)
        self.get_FEs(self.function_evaluations)


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
         "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
         "minmax": "min",
         "obj_func": evaluate_expectation
    }

    model = OriginalNRO(epoch=1000, pop_size=50)
    g_best = model.solve(problem_dict, qubits = k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 04:49:07 PM, INFO, __main__.OriginalNRO: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 04:49:08 PM, INFO, __main__.OriginalNRO: >>>Problem: P, Epoch: 1, Current best: -1.375, Global best: -1.375, Runtime: 0.53316 seconds


150


2024/07/12 04:49:32 PM, INFO, __main__.OriginalNRO: >>>Problem: P, Epoch: 51, Current best: -1.875, Global best: -1.875, Runtime: 0.48363 seconds


7650


KeyboardInterrupt: 

In [98]:
import numpy as np
from mealpy import FloatVar

class ImprovedBSO(Optimizer):
    """
    The improved version: Improved Brain Storm Optimization (IBSO)

    Notes:
        + Remove some probability parameters, and some unnecessary equations.
        + The Levy-flight technique is employed to enhance the algorithm's robustness and resilience in challenging environments.

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + m_clusters (int): [3, 10], number of clusters (m in the paper)
        + p1 (float): 25% percent
        + p2 (float): 50% percent changed by its own (local search), 50% percent changed by outside (global search)
        + p3 (float): 75% percent develop the old idea, 25% invented new idea based on levy-flight
        + p4 (float): [0.4, 0.6], Need more weights on the centers instead of the random position

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, BSO
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = BSO.ImprovedBSO(epoch=1000, pop_size=50, m_clusters=5, p1=0.25, p2=0.5, p3=0.75, p4=0.6)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

    References
    ~~~~~~~~~~
    [1] El-Abd, M. (2017). Global-best brain storm optimization algorithm. Swarm and evolutionary computation, 37, 27-44.
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, m_clusters: int = 5,
                 p1: float = 0.25, p2: float = 0.5, p3: float = 0.75, p4: float = 0.5, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
            m_clusters (int): number of clusters (m in the paper)
            p1 (float): 25% percent
            p2 (float): 50% percent changed by its own (local search), 50% percent changed by outside (global search)
            p3 (float): 75% percent develop the old idea, 25% invented new idea based on levy-flight
            p4 (float): Need more weights on the centers instead of the random position
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [10, 10000])
        self.m_clusters = self.validator.check_int("m_clusters", m_clusters, [2, int(self.pop_size / 5)])
        self.p1 = self.validator.check_float("p1", p1, (0, 1.0))
        self.p2 = self.validator.check_float("p2", p2, (0, 1.0))
        self.p3 = self.validator.check_float("p3", p3, (0, 1.0))
        self.p4 = self.validator.check_float("p4", p4, (0, 1.0))
        self.set_parameters(["epoch", "pop_size", "m_clusters", "p1", "p2", "p3", "p4"])
        self.sort_flag = False
        self.m_solution = int(self.pop_size / self.m_clusters)
        self.pop_group, self.centers = None, None
        self.function_evaluations = 0  # Initialize function evaluations counter

    def count_fe(self):
        """Increment function evaluation counter"""
        self.function_evaluations += 1

    def find_cluster__(self, pop_group):
        centers = []
        for idx in range(0, self.m_clusters):
            local_best = self.get_best_agent(pop_group[idx], self.problem.minmax)
            centers.append(local_best.copy())
        return centers

    def initialization(self):
        if self.pop is None:
            self.pop = self.generate_population(self.pop_size)
        self.pop_group = self.generate_group_population(self.pop, self.m_clusters, self.m_solution)
        self.centers = self.find_cluster__(self.pop_group)

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        epsilon = 1. - 1. * epoch / self.epoch  # 1. Changed here, no need: k
        if self.generator.uniform() < self.p1:  # p_5a
            idx = self.generator.integers(0, self.m_clusters)
            self.centers[idx] = self.generate_agent()
            self.count_fe()  # Increment function evaluation counter
        pop_group = self.pop_group
        for idx in range(0, self.pop_size):  # Generate new individuals
            cluster_id = int(idx / self.m_solution)
            location_id = int(idx % self.m_solution)

            if self.generator.uniform() < self.p2:  # p_6b
                if self.generator.uniform() < self.p3:
                    pos_new = self.centers[cluster_id].solution + epsilon * self.generator.normal(0, 1, self.problem.n_dims)
                else:  # 2. Using levy flight here
                    levy_step = self.get_levy_flight_step(beta=1.0, multiplier=0.001, size=self.problem.n_dims, case=-1)
                    pos_new = self.pop_group[cluster_id][location_id].solution + levy_step
            else:
                id1, id2 = self.generator.choice(range(0, self.m_clusters), 2, replace=False)
                if self.generator.uniform() < self.p4:
                    pos_new = 0.5 * (self.centers[id1].solution + self.centers[id2].solution) + epsilon * self.generator.normal(0, 1, self.problem.n_dims)
                else:
                    rand_id1 = self.generator.integers(0, self.m_solution)
                    rand_id2 = self.generator.integers(0, self.m_solution)
                    pos_new = 0.5 * (self.pop_group[id1][rand_id1].solution + self.pop_group[id2][rand_id2].solution) + \
                              epsilon * self.generator.normal(0, 1, self.problem.n_dims)
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop_group[cluster_id][location_id] = agent
            self.count_fe()  # Increment function evaluation counter
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                pop_group[cluster_id][location_id] = self.get_better_agent(agent, self.pop_group[cluster_id][location_id], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            for idx in range(0, self.m_clusters):
                pop_group[idx] = self.update_target_for_population(pop_group[idx])
                pop_group[idx] = self.greedy_selection_population(self.pop_group[idx], pop_group[idx], self.problem.minmax)

        # Needed to update the centers and population
        self.centers = self.find_cluster__(pop_group)
        self.pop = []
        for idx in range(0, self.m_clusters):
            self.pop += pop_group[idx]
        self.get_FEs(self.function_evaluations)


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = ImprovedBSO(epoch=1000, pop_size=50, m_clusters = 5, p1 = 0.25, p2 = 0.5, p3 = 0.75, p4 = 0.6)
    g_best = model.solve(problem_dict, qubits=k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 04:53:30 PM, INFO, __main__.ImprovedBSO: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 04:53:31 PM, INFO, __main__.ImprovedBSO: >>>Problem: P, Epoch: 1, Current best: -0.84375, Global best: -1.09375, Runtime: 0.16486 seconds


50


2024/07/12 04:53:37 PM, INFO, __main__.ImprovedBSO: >>>Problem: P, Epoch: 51, Current best: -1.5625, Global best: -1.71875, Runtime: 0.11798 seconds


...

37752


2024/07/12 05:09:15 PM, INFO, __main__.ImprovedBSO: >>>Problem: P, Epoch: 801, Current best: -6.9375, Global best: -7.71875, Runtime: 0.18432 seconds


40265


2024/07/12 05:09:24 PM, INFO, __main__.ImprovedBSO: >>>Problem: P, Epoch: 851, Current best: -7.75, Global best: -7.78125, Runtime: 0.18365 seconds


42780
Number of function evaluations:  44285 for qubits:  9
Solution: [-751.98553783 -327.49858946 -437.36207543 -368.27553896 -240.46622547
 -204.83687834 -381.9396261  -707.6713176  -224.26791021 -400.01181614
 -486.53580501 -732.93716479 -343.56933472 -158.62768142 -649.49724806
 -271.71091381 -326.92370947 -217.18673386 -356.72105811 -652.02976057
 -752.41721774 -485.34826474 -632.89033037 -419.53005361 -243.02674658
 -617.17453899 -227.34529788 -541.77806591 -519.80812677 -504.36740342
 -582.77129197 -571.12183166 -353.48547905 -530.45441353 -337.84807688
 -262.18035768], Fitness: -7.90625
Solution: [-751.98553783 -327.49858946 -437.36207543 -368.27553896 -240.46622547
 -204.83687834 -381.9396261  -707.6713176  -224.26791021 -400.01181614
 -486.53580501 -732.93716479 -343.56933472 -158.62768142 -649.49724806
 -271.71091381 -326.92370947 -217.18673386 -356.72105811 -652.02976057
 -752.41721774 -485.34826474 -632.89033037 -419.53005361 -243.02674658
 -617.17453899 -227.34529788 -541

In [100]:
import numpy as np
from mealpy import FloatVar

class DevFBIO(Optimizer):
    """
    The developed : Forensic-Based Investigation Optimization (FBIO)

    Notes:
        + Third loop is removed, the flow and a few equations are improved

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, FBIO
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = FBIO.DevFBIO(epoch=1000, pop_size=50)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.set_parameters(["epoch", "pop_size"])
        self.sort_flag = False
        self.function_evaluations = 0  # Initialize function evaluations counter

    def probability__(self, list_fitness=None):  # Eq.(3) in FBI Inspired Meta-Optimization
        max1 = np.max(list_fitness)
        min1 = np.min(list_fitness)
        return (max1 - list_fitness) / (max1 - min1 + self.EPSILON)

    def count_fe(self):
        """Increment function evaluation counter"""
        self.function_evaluations += 1

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        # Investigation team - team A
        # Step A1
        pop_new = []
        for idx in range(0, self.pop_size):
            n_change = self.generator.integers(0, self.problem.n_dims)
            nb1, nb2 = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}), 2, replace=False)
            # Eq.(2) in FBI Inspired Meta - Optimization
            pos_a = self.pop[idx].solution.copy()
            pos_a[n_change] = self.pop[idx].solution[n_change] + self.generator.normal() * \
                (self.pop[idx].solution[n_change] - (self.pop[nb1].solution[n_change] + self.pop[nb2].solution[n_change]) / 2)
            pos_a = self.correct_solution(pos_a)
            agent = self.generate_empty_agent(pos_a)
            pop_new.append(agent)
            self.count_fe()  # Increment function evaluation counter
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_a)
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_new = self.update_target_for_population(pop_new)
            self.pop = self.greedy_selection_population(self.pop, pop_new, self.problem.minmax)
        list_fitness = np.array([agent.target.fitness for agent in self.pop])
        prob = self.probability__(list_fitness)

        # Step A2
        pop_child = []
        for idx in range(0, self.pop_size):
            if self.generator.random() > prob[idx]:
                r1, r2, r3 = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}), 3, replace=False)
                ## Remove third loop here, the condition also not good, need to remove also. No need Rnd variable
                temp = self.g_best.solution + self.pop[r1].solution + self.generator.uniform() * (self.pop[r2].solution - self.pop[r3].solution)
                condition = self.generator.random(self.problem.n_dims) < 0.5
                pos_new = np.where(condition, temp, self.pop[idx].solution)
            else:
                pos_new = self.problem.generate_solution()
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop_child.append(agent)
            self.count_fe()  # Increment function evaluation counter
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_child = self.update_target_for_population(pop_child)
            self.pop = self.greedy_selection_population(pop_child, self.pop, self.problem.minmax)
        ## Persuing team - team B
        ## Step B1
        pop_new = []
        for idx in range(0, self.pop_size):
            ### Remove third loop here also
            ### Eq.(6) in FBI Inspired Meta-Optimization
            pos_b = self.generator.uniform(0, 1, self.problem.n_dims) * self.pop[idx].solution + \
                    self.generator.uniform(0, 1, self.problem.n_dims) * (self.g_best.solution - self.pop[idx].solution)
            pos_b = self.correct_solution(pos_b)
            agent = self.generate_empty_agent(pos_b)
            pop_new.append(agent)
            self.count_fe()  # Increment function evaluation counter
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_b)
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_new = self.update_target_for_population(pop_new)
            self.pop = self.greedy_selection_population(self.pop, pop_new, self.problem.minmax)
        ## Step B2
        pop_child = []
        for idx in range(0, self.pop_size):
            rr = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}))
            if self.compare_target(self.pop[idx].target, self.pop[rr].target, self.problem.minmax):
                ## Eq.(7) in FBI Inspired Meta-Optimization
                pos_b = self.pop[idx].solution + self.generator.uniform(0, 1, self.problem.n_dims) * \
                        (self.pop[rr].solution - self.pop[idx].solution) + self.generator.uniform() * (self.g_best.solution - self.pop[rr].solution)
            else:
                ## Eq.(8) in FBI Inspired Meta-Optimization
                pos_b = self.pop[idx].solution + self.generator.uniform(0, 1, self.problem.n_dims) * \
                        (self.pop[idx].solution - self.pop[rr].solution) + self.generator.uniform() * (self.g_best.solution - self.pop[idx].solution)
            pos_b = self.correct_solution(pos_b)
            agent = self.generate_empty_agent(pos_b)
            pop_child.append(agent)
            self.count_fe()  # Increment function evaluation counter
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_b)
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_child = self.update_target_for_population(pop_child)
            self.pop = self.greedy_selection_population(pop_child, self.pop, self.problem.minmax)
        self.get_FEs(self.function_evaluations)


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = DevFBIO(epoch=1000, pop_size=50)
    g_best = model.solve(problem_dict, qubits = k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 05:09:56 PM, INFO, __main__.DevFBIO: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 05:09:57 PM, INFO, __main__.DevFBIO: >>>Problem: P, Epoch: 1, Current best: -1.21875, Global best: -1.21875, Runtime: 0.52176 seconds


200


2024/07/12 05:10:07 PM, INFO, __main__.DevFBIO: Solving single objective optimization problem.


...

2024/07/12 05:35:12 PM, INFO, __main__.DevFBIO: >>>Problem: P, Epoch: 601, Current best: -6.8125, Global best: -6.8125, Runtime: 0.74834 seconds


120200


2024/07/12 05:35:49 PM, INFO, __main__.DevFBIO: >>>Problem: P, Epoch: 651, Current best: -6.8125, Global best: -6.8125, Runtime: 0.73929 seconds


130200


KeyboardInterrupt: 

In [17]:
import numpy as np
from mealpy import FloatVar

class OriginalICA(Optimizer):
    """
    The original version of: Imperialist Competitive Algorithm (ICA)

    Links:
        1. https://ieeexplore.ieee.org/document/4425083

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + empire_count (int): [3, 10], Number of Empires (also Imperialists)
        + assimilation_coeff (float): [1.0, 3.0], Assimilation Coefficient (beta in the paper)
        + revolution_prob (float): [0.01, 0.1], Revolution Probability
        + revolution_rate (float): [0.05, 0.2], Revolution Rate       (mu)
        + revolution_step_size (float): [0.05, 0.2], Revolution Step Size  (sigma)
        + zeta (float): [0.05, 0.2], Colonies Coefficient in Total Objective Value of Empires

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, ICA
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = ICA.OriginalICA(epoch=1000, pop_size=50, empire_count = 5, assimilation_coeff = 1.5,
    >>>                         revolution_prob = 0.05, revolution_rate = 0.1, revolution_step_size = 0.1, zeta = 0.1)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

    References
    ~~~~~~~~~~
    [1] Atashpaz-Gargari, E. and Lucas, C., 2007, September. Imperialist competitive algorithm: an algorithm for
    optimization inspired by imperialistic competition. In 2007 IEEE congress on evolutionary computation (pp. 4661-4667). Ieee.
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, empire_count: int = 5, assimilation_coeff: float = 1.5, revolution_prob: float = 0.05,
                 revolution_rate: float = 0.1, revolution_step_size: float = 0.1, zeta: float = 0.1, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size (n: pop_size, m: clusters), default = 100
            empire_count (int): Number of Empires (also Imperialists)
            assimilation_coeff (float): Assimilation Coefficient (beta in the paper)
            revolution_prob (float): Revolution Probability
            revolution_rate (float): Revolution Rate       (mu)
            revolution_step_size (float): Revolution Step Size  (sigma)
            zeta (float): Colonies Coefficient in Total Objective Value of Empires
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [10, 10000])
        self.empire_count = self.validator.check_int("empire_count", empire_count, [2, 2 + int(self.pop_size / 5)])
        self.assimilation_coeff = self.validator.check_float("assimilation_coeff", assimilation_coeff, [1.0, 3.0])
        self.revolution_prob = self.validator.check_float("revolution_prob", revolution_prob, (0, 1.0))
        self.revolution_rate = self.validator.check_float("revolution_rate", revolution_rate, (0, 1.0))
        self.revolution_step_size = self.validator.check_float("revolution_step_size", revolution_step_size, (0, 1.0))
        self.zeta = self.validator.check_float("zeta", zeta, (0, 1.0))
        self.set_parameters(["epoch", "pop_size", "empire_count", "assimilation_coeff", "revolution_prob",
                             "revolution_rate", "revolution_step_size", "zeta"])
        self.sort_flag = True
        self.function_evaluations = 0  # Initialize function evaluations counter

    def count_fe(self):
        """Increment function evaluation counter"""
        self.function_evaluations += 1

    def revolution_country__(self, solution: np.ndarray, n_revoluted: int) -> np.ndarray:
        pos_new = solution + self.revolution_step_size * self.generator.normal(0, 1, self.problem.n_dims)
        idx_list = self.generator.choice(range(0, self.problem.n_dims), n_revoluted, replace=False)
        if len(idx_list) == 0:
            idx_list = np.append(idx_list, self.generator.integers(0, self.problem.n_dims))
        solution[idx_list] = pos_new[idx_list]  # Change only those selected index
        return solution

    def initialization(self):
        if self.pop is None:
            self.pop = self.generate_population(self.pop_size)
        self.pop = self.get_sorted_population(self.pop, self.problem.minmax)
        self.g_best = self.pop[0].copy()
        # Initialization
        self.n_revoluted_variables = int(round(self.revolution_rate * self.problem.n_dims))
        # pop = Empires
        colony_count = self.pop_size - self.empire_count
        self.pop_empires = [agent.copy() for agent in self.pop[:self.empire_count]]
        self.pop_colonies = [agent.copy() for agent in self.pop[self.empire_count:]]

        cost_empires_list = np.array([agent.target.fitness for agent in self.pop_empires])
        cost_empires_list_normalized = cost_empires_list - (np.max(cost_empires_list) + np.min(cost_empires_list))
        prob_empires_list = np.abs(cost_empires_list_normalized / np.sum(cost_empires_list_normalized))
        # Randomly choose colonies to empires
        self.empires = {}
        idx_already_selected = []
        for i in range(0, self.empire_count - 1):
            self.empires[i] = []
            n_colonies = int(round(prob_empires_list[i] * colony_count))
            idx_list = self.generator.choice(list(set(range(0, colony_count)) - set(idx_already_selected)), n_colonies, replace=False).tolist()
            idx_already_selected += idx_list
            for idx in idx_list:
                self.empires[i].append(self.pop_colonies[idx])
        idx_last = list(set(range(0, colony_count)) - set(idx_already_selected))
        self.empires[self.empire_count - 1] = []
        for idx in idx_last:
            self.empires[self.empire_count - 1].append(self.pop_colonies[idx].copy())

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        # Assimilation
        for idx, colonies in self.empires.items():
            for idx_colony, colony in enumerate(colonies):
                pos_new = colony.solution + self.assimilation_coeff * \
                          self.generator.uniform(0, 1, self.problem.n_dims) * (self.pop_empires[idx].solution - colony.solution)
                pos_new = self.correct_solution(pos_new)
                self.empires[idx][idx_colony].solution = pos_new
                if self.mode not in self.AVAILABLE_MODES:
                    self.empires[idx][idx_colony].target = self.get_target(pos_new)
                    self.count_fe()  # Increment function evaluation counter
            self.empires[idx] = self.update_target_for_population(self.empires[idx])
        # Revolution
        for idx, colonies in self.empires.items():
            # Apply revolution to Imperialist
            pos_new_em = self.revolution_country__(self.pop_empires[idx].solution, self.n_revoluted_variables)
            pos_new_em = self.correct_solution(pos_new_em)
            self.pop_empires[idx].solution = pos_new_em
            if self.mode not in self.AVAILABLE_MODES:
                self.pop_empires[idx].target = self.get_target(pos_new_em)
                self.count_fe()  # Increment function evaluation counter
            # Apply revolution to Colonies
            for idx_colony, colony in enumerate(colonies):
                if self.generator.random() < self.revolution_prob:
                    pos_new = self.revolution_country__(colony.solution, self.n_revoluted_variables)
                    pos_new = self.correct_solution(pos_new)
                    self.empires[idx][idx_colony].solution = pos_new
                    if self.mode not in self.AVAILABLE_MODES:
                        self.empires[idx][idx_colony].target = self.get_target(pos_new)
                        self.count_fe()  # Increment function evaluation counter
            self.empires[idx] = self.update_target_for_population(self.empires[idx])
        self.pop_empires = self.update_target_for_population(self.pop_empires)
        self.update_global_best_agent(self.pop_empires, save=False)
        # Intra-Empire Competition
        for idx, colonies in self.empires.items():
            for idx_colony, colony in enumerate(colonies):
                if self.compare_target(colony.target, self.pop_empires[idx].target, self.problem.minmax):
                    self.empires[idx][idx_colony], self.pop_empires[idx] = self.pop_empires[idx].copy(), colony.copy()
        # Update Total Objective Values of Empires
        cost_empires_list = []
        for idx, colonies in self.empires.items():
            fit_list = np.array([agent.target.fitness for agent in colonies])
            fit_empire = self.pop_empires[idx].target.fitness + self.zeta * np.mean(fit_list)
            cost_empires_list.append(fit_empire)
        cost_empires_list = np.array(cost_empires_list)
        # Find possession probability of each empire based on its total power
        cost_empires_list_normalized = cost_empires_list - (np.max(cost_empires_list) + np.min(cost_empires_list))
        prob_empires_list = np.abs(cost_empires_list_normalized / np.sum(cost_empires_list_normalized))  # Vector P
        uniform_list = self.generator.uniform(0, 1, len(prob_empires_list))  # Vector R
        vector_D = prob_empires_list - uniform_list
        idx_empire = np.argmax(vector_D)
        # Find the weakest empire and weakest colony inside it
        idx_weakest_empire = np.argmax(cost_empires_list)
        if len(self.empires[idx_weakest_empire]) > 0:
            colonies_sorted = self.get_sorted_population(self.empires[idx_weakest_empire], self.problem.minmax)
            self.empires[idx_empire].append(colonies_sorted.pop(-1))
        else:
            self.empires[idx_empire].append(self.pop_empires.pop(idx_weakest_empire))

        self.pop = self.pop_empires + self.pop_colonies
        self.get_FEs(self.function_evaluations)


In [18]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = OriginalICA(epoch=300, pop_size=20, empire_count = 5, assimilation_coeff = 1.5,
                            revolution_prob = 0.05, revolution_rate = 0.1, revolution_step_size = 0.1, zeta = 0.1)
    g_best = model.solve(problem_dict, qubits=k)
    print(f"Fitness: {g_best.target.fitness}")

2024/11/29 02:48:46 PM, INFO, __main__.OriginalICA: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/11/29 02:48:47 PM, INFO, __main__.OriginalICA: >>>Problem: P, Epoch: 1, Current best: -0.574609375, Global best: -0.63203125, Runtime: 0.20824 seconds


20


2024/11/29 02:49:00 PM, INFO, __main__.OriginalICA: Solving single objective optimization problem.


...

2024/11/29 02:55:40 PM, INFO, __main__.OriginalICA: >>>Problem: P, Epoch: 151, Current best: -3.9953125, Global best: -3.997265625, Runtime: 2.02355 seconds


15014


2024/11/29 02:57:52 PM, INFO, __main__.OriginalICA: >>>Problem: P, Epoch: 201, Current best: -3.9984375000000005, Global best: -3.9996093749999995, Runtime: 3.12446 seconds


25243


KeyboardInterrupt: 

In [13]:
import numpy as np
from mealpy import FloatVar

class OriginalLCO(Optimizer):
    """
    The original version of: Life Choice-based Optimization (LCO)

    Links:
        1. https://doi.org/10.1007/s00500-019-04443-z

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + r1 (float): [1.5, 4], coefficient factor, default = 2.35

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, LCO
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = LCO.OriginalLCO(epoch=1000, pop_size=50, r1 = 2.35)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

    References
    ~~~~~~~~~~
    [1] Khatri, A., Gaba, A., Rana, K.P.S. and Kumar, V., 2020. A novel life choice-based optimizer. Soft Computing, 24(12), pp.9121-9141.
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, r1: float = 2.35, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
            r1 (float): coefficient factor
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.r1 = self.validator.check_float("r1", r1, [1.0, 3.0])
        self.set_parameters(["epoch", "pop_size", "r1"])
        self.n_agents = int(np.ceil(np.sqrt(self.pop_size)))
        self.sort_flag = True
        self.function_evaluations = 0  # Initialize function evaluations counter

    def count_fe(self):
        """Increment function evaluation counter"""
        self.function_evaluations += 1

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        pop_new = []
        for idx in range(0, self.pop_size):
            prob = self.generator.random()
            if prob > 0.875:  # Update using Eq. 1, update from n best position
                temp = np.array([self.generator.random() * self.pop[j].solution for j in range(0, self.n_agents)])
                temp = np.mean(temp, axis=0)
            elif prob < 0.7:  # Update using Eq. 2-6
                f1 = 1 - epoch / self.epoch
                f2 = 1 - f1
                prev_pos = self.g_best.solution if idx == 0 else self.pop[idx-1].solution
                best_diff = f1 * self.r1 * (self.g_best.solution - self.pop[idx].solution)
                better_diff = f2 * self.r1 * (prev_pos - self.pop[idx].solution)
                temp = self.pop[idx].solution + self.generator.random() * better_diff + self.generator.random() * best_diff
            else:
                temp = self.problem.ub - (self.pop[idx].solution - self.problem.lb) * self.generator.random()
            pos_new = self.correct_solution(temp)
            agent = self.generate_empty_agent(pos_new)
            pop_new.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.count_fe()  # Increment function evaluation counter
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_new = self.update_target_for_population(pop_new)
            self.pop = self.greedy_selection_population(self.pop, pop_new, self.problem.minmax)
        self.get_FEs(self.function_evaluations)


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(1000.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = OriginalLCO(epoch=1000, pop_size=20, r1 = 2.35)
    g_best = model.solve(problem_dict, qubits = k)
    print(f"Fitness: {g_best.target.fitness}")

2024/11/29 02:45:34 PM, INFO, __main__.OriginalLCO: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/11/29 02:45:34 PM, INFO, __main__.OriginalLCO: >>>Problem: P, Epoch: 1, Current best: -1.0113207547169811, Global best: -1.0113207547169811, Runtime: 0.14163 seconds


20


2024/11/29 02:45:41 PM, INFO, __main__.OriginalLCO: >>>Problem: P, Epoch: 51, Current best: -1.788679245283019, Global best: -1.788679245283019, Runtime: 0.11735 seconds


...

2024/11/29 02:47:44 PM, INFO, __main__.OriginalLCO: >>>Problem: P, Epoch: 851, Current best: -2.5066037735849056, Global best: -2.5066037735849056, Runtime: 0.11292 seconds


17020


2024/11/29 02:47:50 PM, INFO, __main__.OriginalLCO: >>>Problem: P, Epoch: 901, Current best: -2.5066037735849056, Global best: -2.5066037735849056, Runtime: 0.13509 seconds


18020


KeyboardInterrupt: 

In [107]:
import numpy as np
from mealpy import FloatVar

class OriginalCRO(Optimizer):
    """
    The original version of: Coral Reefs Optimization (CRO)

    Links:
        1. https://downloads.hindawi.com/journals/tswj/2014/739768.pdf

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + po (float): [0.2, 0.5], the rate between free/occupied at the beginning
        + Fb (float): [0.6, 0.9], BroadcastSpawner/ExistingCorals rate
        + Fa (float): [0.05, 0.3], fraction of corals duplicates its self and tries to settle in a different part of the reef
        + Fd (float): [0.05, 0.5], fraction of the worse health corals in reef will be applied depredation
        + Pd (float): [0.1, 0.7], Probability of depredation
        + GCR (float): [0.05, 0.2], probability for mutation process
        + gamma_min (float): [0.01, 0.1] factor for mutation process
        + gamma_max (float): [0.1, 0.5] factor for mutation process
        + n_trials (int): [2, 10], number of attempts for a larvar to set in the reef.

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, CRO
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = CRO.OriginalCRO(epoch=1000, pop_size=50, po = 0.4, Fb = 0.9, Fa = 0.1, Fd = 0.1, Pd = 0.5, GCR = 0.1, gamma_min = 0.02, gamma_max = 0.2, n_trials = 5)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

    References
    ~~~~~~~~~~
    [1] Salcedo-Sanz, S., Del Ser, J., Landa-Torres, I., Gil-López, S. and Portilla-Figueras, J.A., 2014.
    The coral reefs optimization algorithm: a novel metaheuristic for efficiently solving optimization problems. The Scientific World Journal, 2014.
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, po: float = 0.4, Fb: float = 0.9, Fa: float = 0.1, Fd: float = 0.1,
                 Pd: float = 0.5, GCR: float = 0.1, gamma_min: float = 0.02, gamma_max: float = 0.2, n_trials: int = 3, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
            po (float): the rate between free/occupied at the beginning
            Fb (float): BroadcastSpawner/ExistingCorals rate
            Fa (float): fraction of corals duplicates its self and tries to settle in a different part of the reef
            Fd (float): fraction of the worse health corals in reef will be applied depredation
            Pd (float): the maximum of probability of depredation
            GCR (float): probability for mutation process
            gamma_min (float): factor for mutation process
            gamma_max (float): factor for mutation process
            n_trials (int): number of attempts for a larva to set in the reef.
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])  # ~ number of space
        self.po = self.validator.check_float("po", po, (0, 1.0))
        self.Fb = self.validator.check_float("Fb", Fb, (0, 1.0))
        self.Fa = self.validator.check_float("Fa", Fa, (0, 1.0))
        self.Fd = self.validator.check_float("Fd", Fd, (0, 1.0))
        self.Pd = self.validator.check_float("Pd", Pd, (0, 1.0))
        self.GCR = self.validator.check_float("GCR", GCR, (0, 1.0))
        self.gamma_min = self.validator.check_float("gamma_min", gamma_min, (0, 0.15))
        self.gamma_max = self.validator.check_float("gamma_max", gamma_max, (0.15, 1.0))
        self.n_trials = self.validator.check_int("n_trials", n_trials, [2, int(self.pop_size / 2)])
        self.set_parameters(["epoch", "pop_size", "po", "Fb", "Fa", "Fd", "Pd", "GCR", "gamma_min", "gamma_max", "n_trials"])
        self.sort_flag = False
        self.function_evaluations = 0

    def count_fe(self):
        self.function_evaluations += 1

    def initialization(self):
        if self.pop is None:
            self.pop = self.generate_population(self.pop_size)
        self.reef = np.array([])
        self.occupied_position = []  # after a gen, you should update the occupied_position
        self.G1 = self.gamma_max
        self.alpha = 10 * self.Pd / self.epoch
        self.gama = 10 * (self.gamma_max - self.gamma_min) / self.epoch
        self.num_occupied = int(self.pop_size / (1 + self.po))
        self.dyn_Pd = 0
        self.occupied_list = np.zeros(self.pop_size)
        self.occupied_idx_list = self.generator.choice(list(range(self.pop_size)), self.num_occupied, replace=False)
        self.occupied_list[self.occupied_idx_list] = 1

    def gaussian_mutation__(self, position):
        random_pos = position + self.G1 * (self.problem.ub - self.problem.lb) * self.generator.normal(0, 1, self.problem.n_dims)
        condition = self.generator.random(self.problem.n_dims) < self.GCR
        pos_new = np.where(condition, random_pos, position)
        return self.correct_solution(pos_new)

    ### Crossover
    def multi_point_cross__(self, pos1, pos2):
        p1, p2 = self.generator.choice(list(range(len(pos1))), 2, replace=False)
        start, end = min(p1, p2), max(p1, p2)
        pos_new = np.concatenate((pos1[:start], pos2[start:end], pos1[end:]), axis=0)
        return self.correct_solution(pos_new)

    def larvae_setting__(self, larvae):
        # Trial to land on a square of reefs
        for larva in larvae:
            for idx in range(self.n_trials):
                pdx = self.generator.integers(0, self.pop_size - 1)
                if self.occupied_list[pdx] == 0:
                    self.pop[pdx] = larva
                    self.occupied_idx_list = np.append(self.occupied_idx_list, pdx)  # Update occupied id
                    self.occupied_list[pdx] = 1  # Update occupied list
                    break
                else:
                    if self.compare_target(larva.target, self.pop[pdx].target, self.problem.minmax):
                        self.pop[pdx] = larva
                        break

    def sort_occupied_reef__(self):
        def reef_fitness(idx):
            return self.pop[idx].target.fitness
        return sorted(self.occupied_idx_list, key=reef_fitness)

    def broadcast_spawning_brooding__(self):
        # Step 1a
        larvae = []
        selected_corals = self.generator.choice(self.occupied_idx_list, int(len(self.occupied_idx_list) * self.Fb), replace=False)
        for idx in self.occupied_idx_list:
            if idx not in selected_corals:
                pos_new = self.gaussian_mutation__(self.pop[idx].solution)
                agent = self.generate_empty_agent(pos_new)
                larvae.append(agent)
                if self.mode not in self.AVAILABLE_MODES:
                    larvae[-1].target = self.get_target(pos_new)
                    self.count_fe()  # Increment function evaluations counter
        # Step 1b
        while len(selected_corals) >= 2:
            id1, id2 = self.generator.choice(range(len(selected_corals)), 2, replace=False)
            pos_new = self.multi_point_cross__(self.pop[selected_corals[id1]].solution, self.pop[selected_corals[id2]].solution)
            agent = self.generate_empty_agent(pos_new)
            larvae.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                larvae[-1].target = self.get_target(pos_new)
                self.count_fe()  # Increment function evaluations counter
            selected_corals = np.delete(selected_corals, [id1, id2])
        return self.update_target_for_population(larvae)

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        ## Broadcast Spawning Brooding
        larvae = self.broadcast_spawning_brooding__()
        self.larvae_setting__(larvae)
        ## Asexual Reproduction
        num_duplicate = int(len(self.occupied_idx_list) * self.Fa)
        pop_best = [self.pop[idx] for idx in self.occupied_idx_list]
        pop_best = self.get_sorted_and_trimmed_population(pop_best, num_duplicate, self.problem.minmax)
        for larva in pop_best:
            self.count_fe()  # Increment function evaluations counter
        self.larvae_setting__(pop_best)
        ## Depredation
        if self.generator.random() < self.dyn_Pd:
            num__depredation__ = int(len(self.occupied_idx_list) * self.Fd)
            idx_list_sorted = self.sort_occupied_reef__()
            selected_depredator = idx_list_sorted[-num__depredation__:]
            self.occupied_idx_list = np.setdiff1d(self.occupied_idx_list, selected_depredator)
            for idx in selected_depredator:
                self.occupied_list[idx] = 0
        if self.dyn_Pd <= self.Pd:
            self.dyn_Pd += self.alpha
        if self.G1 >= self.gamma_min:
            self.G1 -= self.gama
        self.get_FEs(self.function_evaluations)


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={ "shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = OriginalCRO(epoch=1500, pop_size=50, po = 0.4, Fb = 0.9, Fa = 0.1, Fd = 0.1, Pd = 0.5, GCR = 0.1, gamma_min = 0.02, gamma_max = 0.2, n_trials = 5)
    g_best = model.solve(problem_dict, qubits=k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

ansatz_num_parameters= 12


2024/07/12 06:24:00 PM, INFO, __main__.OriginalCRO: Solving single objective optimization problem.
2024/07/12 06:24:01 PM, INFO, __main__.OriginalCRO: >>>Problem: P, Epoch: 1, Current best: -1.15625, Global best: -1.15625, Runtime: 0.12498 seconds


23


2024/07/12 06:24:06 PM, INFO, __main__.OriginalCRO: >>>Problem: P, Epoch: 51, Current best: -1.8125, Global best: -1.8125, Runtime: 0.07555 seconds


1560


...

2024/07/12 06:25:07 PM, INFO, __main__.OriginalCRO: >>>Problem: P, Epoch: 651, Current best: -2.8125, Global best: -2.8125, Runtime: 0.08416 seconds


19546


2024/07/12 06:25:11 PM, INFO, __main__.OriginalCRO: >>>Problem: P, Epoch: 701, Current best: -2.8125, Global best: -2.8125, Runtime: 0.10749 seconds


21025


KeyboardInterrupt: 

In [110]:
import numpy as np
from mealpy import FloatVar

class OriginalFPA(Optimizer):
    """
    The original version of: Flower Pollination Algorithm (FPA)

    Links:
        1. https://doi.org/10.1007/978-3-642-32894-7_27

    Hyper-parameters should fine-tune in approximate range to get faster convergence toward the global optimum:
        + p_s (float): [0.5, 0.95], switch probability, default = 0.8
        + levy_multiplier: [0.0001, 1000], mutiplier factor of Levy-flight trajectory, depends on the problem

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, FPA
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>> }
    >>>
    >>> model = FPA.OriginalFPA(epoch=1000, pop_size=50, p_s = 0.8, levy_multiplier = 0.2)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

    References
    ~~~~~~~~~~
    [1] Yang, X.S., 2012, September. Flower pollination algorithm for global optimization. In International
    conference on unconventional computing and natural computation (pp. 240-249). Springer, Berlin, Heidelberg.
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, p_s: float = 0.8, levy_multiplier: float = 0.1, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
            p_s (float): switch probability, default = 0.8
            levy_multiplier (float): multiplier factor of Levy-flight trajectory, default = 0.2
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.p_s = self.validator.check_float("p_s", p_s, (0, 1.0))
        self.levy_multiplier = self.validator.check_float("levy_multiplier", levy_multiplier, (-10000, 10000))
        self.set_parameters(["epoch", "pop_size", "p_s", "levy_multiplier"])
        self.sort_flag = False
        self.function_evaluations = 0

    def count_fe(self):
        self.function_evaluations += 1

    def amend_solution(self, solution: np.ndarray) -> np.ndarray:
        condition = np.logical_and(self.problem.lb <= solution, solution <= self.problem.ub)
        random_pos = self.problem.generate_solution()
        return np.where(condition, solution, random_pos)

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        pop = []
        for idx in range(0, self.pop_size):
            if self.generator.uniform() < self.p_s:
                levy = self.get_levy_flight_step(multiplier=self.levy_multiplier, size=self.problem.n_dims, case=-1)
                pos_new = self.pop[idx].solution + 1.0 / np.sqrt(epoch) * levy * (self.pop[idx].solution - self.g_best.solution)
            else:
                id1, id2 = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}), 2, replace=False)
                pos_new = self.pop[idx].solution + self.generator.uniform() * (self.pop[id1].solution - self.pop[id2].solution)
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.count_fe()  # Increment function evaluations counter
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop = self.update_target_for_population(pop)
            self.pop = self.greedy_selection_population(self.pop, pop, self.problem.minmax)
        self.get_FEs(self.function_evaluations)


In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = OriginalFPA(epoch=1000, pop_size=50, p_s = 0.8, levy_multiplier = 0.2)
    g_best = model.solve(problem_dict, qubits=k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 06:25:21 PM, INFO, __main__.OriginalFPA: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 06:25:21 PM, INFO, __main__.OriginalFPA: >>>Problem: P, Epoch: 1, Current best: -0.96875, Global best: -0.96875, Runtime: 0.14361 seconds


50


2024/07/12 06:25:26 PM, INFO, __main__.OriginalFPA: Solving single objective optimization problem.


...

2024/07/12 06:31:25 PM, INFO, __main__.OriginalFPA: >>>Problem: P, Epoch: 901, Current best: -5.84375, Global best: -5.84375, Runtime: 0.17238 seconds


45050


2024/07/12 06:31:33 PM, INFO, __main__.OriginalFPA: >>>Problem: P, Epoch: 951, Current best: -5.84375, Global best: -5.84375, Runtime: 0.17051 seconds


47550
Number of function evaluations:  49950 for qubits:  7
Solution: [-7.08969776  1.52689325 -0.86246173  3.21915903  5.4679884   2.28594227
  2.02833451 -4.13679018 -3.87197162 -1.72768835  4.98163832 -4.15663258
  2.26523858  1.33270941  4.93079162  0.75733245 -4.81529433  7.48637561
 -4.60211731 -4.9917928   4.79578823 -1.75533496  5.15466435  7.80946864
 -0.55793826 -4.58282126 -8.03097463 -4.50394927  4.02420877 -8.86073679], Fitness: -5.875
Solution: [-7.08969776  1.52689325 -0.86246173  3.21915903  5.4679884   2.28594227
  2.02833451 -4.13679018 -3.87197162 -1.72768835  4.98163832 -4.15663258
  2.26523858  1.33270941  4.93079162  0.75733245 -4.81529433  7.48637561
 -4.60211731 -4.9917928   4.79578823 -1.75533496  5.15466435  7.80946864
 -0.55793826 -4.58282126 -8.03097463 -4.50394927  4.02420877 -8.86073679], Fitness: -5.875
ansatz_num_parameters= 32


ValueError: The number of values (30) does not match the number of parameters (32) for the 0-th circuit.

In [112]:
import numpy as np
from mealpy import FloatVar

class OriginalRUN(Optimizer):
    """
    The original version of: RUNge Kutta optimizer (RUN)

    Links:
        1. https://doi.org/10.1016/j.eswa.2021.115079
        2. https://imanahmadianfar.com/codes/
        3. https://www.aliasgharheidari.com/RUN.html

    Examples
    ~~~~~~~~
    >>> import numpy as np
    >>> from mealpy import FloatVar, RUN
    >>>
    >>> def objective_function(solution):
    >>>     return np.sum(solution**2)
    >>>
    >>> problem_dict = {
    >>>     "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    >>>     "minmax": "min",
    >>>     "obj_func": objective_function
    >>>
    >>> model = RUN.OriginalRUN(epoch=1000, pop_size=50)
    >>> g_best = model.solve(problem_dict)
    >>> print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    >>> print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

    References
    ~~~~~~~~~~
    [1] Ahmadianfar, I., Heidari, A. A., Gandomi, A. H., Chu, X., & Chen, H. (2021). RUN beyond the metaphor: An efficient
    optimization algorithm based on Runge Kutta method. Expert Systems with Applications, 181, 115079.
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.set_parameters(["epoch", "pop_size"])
        self.is_parallelizable = False
        self.sort_flag = False
        self.function_evaluations = 0

    def count_fe(self):
        self.function_evaluations += 1

    def runge_kutta__(self, xb, xw, delta_x):
        dim = len(xb)
        C = self.generator.integers(1, 3) * (1 - self.generator.random())
        r1 = self.generator.random(dim)
        r2 = self.generator.random(dim)
        K1 = 0.5 * (self.generator.random() * xw - C * xb)
        K2 = 0.5 * (self.generator.random() * (xw + r2*K1*delta_x/2) - (C*xb + r1*K1*delta_x/2))
        K3 = 0.5 * (self.generator.random() * (xw + r2*K2*delta_x/2) - (C*xb + r1*K2*delta_x/2))
        K4 = 0.5 * (self.generator.random() * (xw + r2*K3*delta_x) - (C*xb + r1*K3*delta_x))
        return (K1 + 2*K2 + 2*K3 + K4)/6

    def uniform_random__(self, a, b, size):
        a2, b2 = a/2, b/2
        mu = a2 + b2
        sig = b2 - a2
        return mu + sig * (2 * self.generator.uniform(0, 1, size) - 1)

    def get_index_of_best_agent__(self, pop):
        fit_list = np.array([agent.target.fitness for agent in pop])
        if self.problem.minmax == "min":
            return np.argmin(fit_list)
        else:
            return np.argmax(fit_list)

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        f = 20 * np.exp(-(12. * epoch / self.epoch))        # Eq.17.6
        SF = 2.*(0.5 - self.generator.random(self.pop_size)) * f     # Eq.17.5
        x_list = np.array([agent.solution for agent in self.pop])
        x_average = np.mean(x_list, axis=0)     # Determine the Average of Solutions
        for idx in range(0, self.pop_size):
            ## Determine Delta X (Eqs. 11.1 to 11.3)
            gama = self.generator.random() * (self.pop[idx].solution - self.generator.uniform(0, 1, self.problem.n_dims) *
                                       (self.problem.ub - self.problem.lb)) * np.exp(-4 * epoch / self.epoch)
            stp = self.generator.uniform(0, 1, self.problem.n_dims) * ((self.g_best.solution - self.generator.random() * x_average) + gama)
            delta_x = 2 * self.generator.uniform(0, 1, self.problem.n_dims) * np.abs(stp)
            ## Determine Three Random Indices of Solutions
            a, b, c = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}), 3, replace=False)
            id_min_x = self.get_index_of_best_agent__([self.pop[a], self.pop[b], self.pop[c]])
            ## Determine Xb and Xw for using in Runge Kutta method
            if self.compare_target(self.pop[idx].target, self.pop[id_min_x].target, self.problem.minmax):
                xb, xw = self.pop[idx].solution, self.pop[id_min_x].solution
            else:
                xb, xw = self.pop[id_min_x].solution, self.pop[idx].solution
            ## Search Mechanism (SM) of RUN based on Runge Kutta Method
            SM = self.runge_kutta__(xb, xw, delta_x)
            local_best = self.get_best_agent(self.pop, self.problem.minmax)
            L = self.generator.choice(range(0, 2), self.problem.n_dims)
            xc = L * self.pop[idx].solution + (1 - L) * self.pop[a].solution        # Eq. 17.3
            xm = L * self.g_best.solution + (1 - L) * local_best.solution           # Eq. 17.4
            r = self.generator.choice([1, -1], self.problem.n_dims)          # An integer number
            g = 2 * self.generator.random()
            mu = 0.5 + 1 * self.generator.uniform(0, 1, self.problem.n_dims)
            ## Determine New Solution Based on Runge Kutta Method (Eq.18)
            if self.generator.random() < 0.5:
                pos_new = xc + r * SF[idx] * g * xc + SF[idx] * SM + mu * (xm - xc)
            else:
                pos_new = xm + r * SF[idx] * g * xm + SF[idx] * SM + mu * (self.pop[a].solution - self.pop[b].solution)
            pos_new = self.correct_solution(pos_new)
            tar_new = self.get_target(pos_new)
            self.count_fe()  # Increment function evaluations counter
            if self.compare_target(tar_new, self.pop[idx].target, self.problem.minmax):
                self.pop[idx].update(solution=pos_new, target=tar_new)
            ## Enhanced solution quality (ESQ)  (Eq. 19)
            if self.generator.random() < 0.5:
                w = self.uniform_random__(0, 2, self.problem.n_dims) * np.exp(-5*self.generator.random() * epoch / self.epoch)        # Eq.19-1
                r = np.floor(self.uniform_random__(-1, 2, 1))
                u = 2 * self.generator.random(self.problem.n_dims)
                a, b, c = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}), 3, replace=False)
                x_ave = (self.pop[a].solution + self.pop[b].solution + self.pop[c].solution) / 3                # Eq.19-2
                beta = self.generator.random(self.problem.n_dims)
                x_new1 = beta * self.g_best.solution + (1 - beta) * x_ave                                               # Eq.19-3
                x_new2_temp1 = x_new1 + r*w * np.abs(self.generator.normal(0, 1, self.problem.n_dims) + (x_new1 - x_ave))
                x_new2_temp2 = x_new1 - x_ave + r*w*np.abs(self.generator.normal(0, 1, self.problem.n_dims) + u * x_new1 - x_ave)
                x_new2 = np.where(w < 1, x_new2_temp1, x_new2_temp2)
                pos_new2 = self.correct_solution(x_new2)
                tar_new2 = self.get_target(pos_new2)
                self.count_fe()  # Increment function evaluations counter
                if self.compare_target(tar_new2, self.pop[idx].target, self.problem.minmax):
                    self.pop[idx].update(solution=pos_new2, target=tar_new2)
                else:
                    if w[self.generator.integers(0, self.problem.n_dims)] > self.generator.random():
                        SM = self.runge_kutta__(self.pop[idx].solution, pos_new2, delta_x)
                        x_new3 = pos_new2 - self.generator.random()*pos_new2 + \
                                 SF[idx] * (SM + (2 * self.generator.random(self.problem.n_dims)*self.g_best.solution - pos_new2))       # Eq. 20
                        pos_new3 = self.correct_solution(x_new3)
                        tar_new3 = self.get_target(pos_new3)
                        self.count_fe()  # Increment function evaluations counter
                        if self.compare_target(tar_new3, self.pop[idx].target, self.problem.minmax):
                            self.pop[idx].update(solution=pos_new3, target=tar_new3)
        self.get_FEs(self.function_evaluations)

In [ ]:
min_qubits =3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar

    problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
    }

    model = OriginalRUN(epoch=1000, pop_size=50)
    g_best = model.solve(problem_dict, qubits = k)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/12 06:38:25 PM, INFO, __main__.OriginalRUN: Solving single objective optimization problem.


ansatz_num_parameters= 12


2024/07/12 06:38:26 PM, INFO, __main__.OriginalRUN: >>>Problem: P, Epoch: 1, Current best: -1.0625, Global best: -1.0625, Runtime: 0.21998 seconds


86


2024/07/12 06:38:34 PM, INFO, __main__.OriginalRUN: Solving single objective optimization problem.


...

2024/07/12 06:42:06 PM, INFO, __main__.OriginalRUN: >>>Problem: P, Epoch: 151, Current best: -2.375, Global best: -2.375, Runtime: 0.32700 seconds


13837


2024/07/12 06:42:21 PM, INFO, __main__.OriginalRUN: >>>Problem: P, Epoch: 201, Current best: -2.375, Global best: -2.375, Runtime: 0.31968 seconds


18352


KeyboardInterrupt: 